# CRRT / TMP — analysis notebook

*Cumulative operating time and early transmembrane pressure surge in relation to mortality during continuous renal replacement therapy* (Scientific Reports, under revision)

See `README.md` for what this notebook does and how it is organized.


## Setup


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from common import *  # noqa: F401,F403


## 1. Primary analysis: cumulative operating time (Tier 1)


In [ ]:
CAP_INHOSP = 90 * 24
c["end_inhosp_h_capped"] = c["end_inhosp_h"].clip(upper=CAP_INHOSP)
c["death_inhosp_capped"] = c["death_inhosp"].where(c["end_inhosp_h"] <= CAP_INHOSP, 0)
print(f"in-hospital deaths before 90-day cap {c.death_inhosp.sum()} -> after {c.death_inhosp_capped.sum()}")
print(f"end_inhosp_h max: {c.end_inhosp_h.max()/24:.0f}d -> capped {c.end_inhosp_h_capped.max()/24:.0f}d")

_t0map = c.set_index("stay_id")["t0"]

# Prepared datape = pd.read_parquet(CRRT_PROC_PARQUET)
iv = pd.read_parquet(CRRT_INPUT_PARQUET)
pe = pe[pe["stay_id"].isin(set(c["stay_id"]))].copy()
iv = iv[iv["stay_id"].isin(set(c["stay_id"]))].copy()
print(f"raw operating-time records: procedure {len(pe)}/{pe['stay_id'].nunique()} pts, "
      f"input events {len(iv)}/{iv['stay_id'].nunique()} pts")

print("\n" + "=" * 84)
print("  MIMIC Tier 1, cumulative operating time (per 24h) -> mortality: 12 settings (source x gap x endpoint)")
print("=" * 84)
print(f"  {'source':<14}{'gap':>5}{'endpoint':>10}{'HR':>9}{'95%CI':>18}{'p':>9}{'N':>7}{'ev':>6}")
SOURCES = {"procedure": pe, "input_union": iv}
ENDPOINTS = [("28d", "end28_h", "event28"), ("in-hosp", "end_inhosp_h_capped", "death_inhosp_capped")]
rows = []; seg_cache = {}
for sname, raw in SOURCES.items():
    for gap in [3, 6, 12]:
        segs = build_segments(raw, _t0map, gap); seg_cache[(sname, gap)] = segs
        for ename, ecol, evcol in ENDPOINTS:
            _mask = c[ecol].notna() & (c[ecol] > 0)
            d = c[_mask].reset_index(drop=True)
            hr, lo, hi, p, n, ev = fit_grid_cox(d, XC_C, segs, CONT_M, BIN_M, ecol, evcol,
                                                builder="mimic", scale=24, mask=_mask.values)
            star = "*" if (pd.notna(p) and p < 0.05) else ""
            print(f"  {sname:<14}{gap:>4}h{ename:>10}{hr:>9.3f}  [{lo:.3f},{hi:.3f}]"
                  f"{p:>8.3f}{star}{n:>7}{ev:>6}")
            rows.append([sname, gap, ename, hr, lo, hi, p, n, ev])
r_l1m = pd.DataFrame(rows, columns=["source", "gap", "endpoint", "HR", "lo", "hi", "p", "N", "ev"])
print("=" * 84)
okm = r_l1m.dropna(subset=["HR"])
print(f"  summary: {len(okm)} settings, HR range [{okm.HR.min():.3f},{okm.HR.max():.3f}], "
      f"max CI upper {okm.hi.max():.3f}, min p {okm.p.min():.3f}")
print(f"  -> all settings HR~1.00, significant {int((okm.p<0.05).sum())}/{len(okm)}")

SEG_MAIN = seg_cache[("procedure", 6)]

print("\n" + "-" * 84)
print("  [Tier 1 null, robustness] precision argument (BIC-BF unsuitable for continuous TVC)")
print("-" * 84)
_w = okm[okm.endpoint == "28d"]
print(f"  28d, 6 settings: HR {_w.HR.min():.3f}-{_w.HR.max():.3f}, max CI upper {_w.hi.max():.3f}, "
      f"event {int(_w.ev.iloc[0])}")
print(f"  -> per-24h HR brackets 1 tightly, [{_w.lo.min():.3f}, {_w.hi.max():.3f}].")
print(f"     Negligible cumulative effect even at 28d (672h); 753 events give adequate power -> confirmatory null.")
print(f"     (Contrast: Tier 2 surge CI [1.01,2.07] is wide and borderline; quantified below with a Bayes factor.)")

_tot = pd.Series({sid: sum(e - s for s, e in segs) for sid, segs in SEG_MAIN.items()})
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].hist(_tot.clip(upper=720) / 24, bins=60, color="#34495e")
ax[0].set_xlabel("Cumulative CRRT duration (days, clip 30)"); ax[0].set_ylabel("patients")
ax[0].set_title(f"Cumulative duration (procedure, gap6h, median={_tot.median()/24:.1f}d)",
                fontsize=10, loc="left")
lab = [f"{s[:4]}/{g}h/{e}" for s, g, e in zip(okm.source, okm.gap, okm.endpoint)]
y = np.arange(len(okm))[::-1]
ax[1].errorbar(okm.HR, y, xerr=[okm.HR - okm.lo, okm.hi - okm.HR], fmt="o", color="#2c3e50",
               ecolor="#7f8c8d", capsize=3, lw=1, ms=5)
ax[1].axvline(1.0, ls="--", color="#c0392b", lw=1)
ax[1].set_yticks(y); ax[1].set_yticklabels(lab, fontsize=8)
ax[1].set_xlabel("HR per 24h (95% CI)"); ax[1].set_xlim(0.985, 1.015)
ax[1].set_title("MIMIC Tier1: 12 settings", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "l1_mimic.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  saved: {FIGDIR/'l1_mimic.png'}")


In [ ]:
# Prepared upstream (cohort-construction step, not part of this notebook): the
# eICU CRRT cohort with covariates already attached (identification from
# treatment/intakeOutput/patient/apacheApsVar/apachePatientResult/lab/infusionDrug,
# and the I/O-proxy operating-time fields). Columns and derivation are listed in
# README.md "Prepared inputs".
base_e = pd.read_parquet(EICU_COHORT_PARQUET)

CONT_E_use = [x for x in CONT_E if x in base_e.columns]
BIN_E_use = [x for x in BIN_E if x in base_e.columns]
for bcol in BIN_E_use:
    base_e[bcol] = pd.to_numeric(base_e[bcol], errors="coerce").fillna(0).astype(int)
base_e = base_e.reset_index(drop=True)
print(f"eICU CRRT cohort loaded: {len(base_e)} patients "
      f"(dialysis records n_rec>=1: {int((base_e['n_rec']>=1).sum())})")

XC_E = []
for mi in range(M_IMP):
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=SEED + mi,
                           sample_posterior=True, initial_strategy="median")
    XC_E.append(pd.DataFrame(imp.fit_transform(base_e[CONT_E_use]), columns=CONT_E_use,
                             index=base_e.index))

print("\n" + "=" * 78)
print("  eICU Tier 1, cumulative operating time (per 24h) -> in-hospital mortality: 4 settings (min records x termination)")
print("=" * 78)
rows = []
for min_rec in [2, 3]:
    for term_mode in ["a_last", "b_plus1gap"]:
        _m = base_e["n_rec"] >= min_rec
        d = base_e[_m].copy()
        if term_mode == "a_last":
            d["term_h"] = (d["term_off"] - d["t0_off"]) / 60.0
        else:
            d["term_h"] = ((d["term_off"] - d["t0_off"]) + d["med_gap_min"].fillna(0)) / 60.0
        d["term_h"] = np.minimum(d["term_h"], d["disch_h"])
        n0 = int((d["term_h"] <= 0).sum())
        if n0: print(f"    (excluded: operating time<=0 for {n0} patients, I/O-proxy could not be computed)")
        _keepmask = _m.copy(); _keepmask[_m] = (d["term_h"] > 0).values
        d = d[d["term_h"] > 0].reset_index(drop=True)
        XC_sub = [Xc[_keepmask.values].reset_index(drop=True) for Xc in XC_E]
        hr, lo, hi, p, n, ev = fit_grid_cox(d, XC_sub, None, CONT_E_use, BIN_E_use,
                                            "end_h", "event", builder="eicu", scale=24)
        star = "*" if (pd.notna(p) and p < 0.05) else ""
        print(f"  records>={min_rec}, term={term_mode:<11} HR={hr:.3f} [{lo:.3f},{hi:.3f}] "
              f"p={p:.3f}{star} (N={n},ev={ev})")
        rows.append([min_rec, term_mode, hr, lo, hi, p, n, ev])
r_l1e = pd.DataFrame(rows, columns=["min_rec", "term", "HR", "lo", "hi", "p", "N", "ev"])
print("=" * 78)
oke = r_l1e.dropna(subset=["HR"])
print(f"  summary: {len(oke)} settings, HR range [{oke.HR.min():.3f},{oke.HR.max():.3f}], min p {oke.p.min():.3f}")
print(f"  -> agrees with MIMIC (operating time unrelated to mortality) -> multicenter triangulation")
print(f"  limitations: operating time is a dialysis I/O proxy (coarser than MIMIC); aps is a SOFA proxy; heparin is an infusion flag")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
_d2 = base_e[base_e["n_rec"] >= 2].copy()
_d2["dur_d"] = ((_d2["term_off"] - _d2["t0_off"]) / 60.0 / 24).clip(upper=30)
ax[0].hist(_d2["dur_d"], bins=50, color="#8e44ad")
ax[0].set_xlabel("eICU cumulative duration (days, clip 30)"); ax[0].set_ylabel("patients")
ax[0].set_title(f"eICU duration proxy (n_rec>=2, median={_d2['dur_d'].median():.1f}d)",
                fontsize=10, loc="left")
labs = [f"M:{s[:4]}/{g}h/{e}" for s, g, e in zip(okm.source, okm.gap, okm.endpoint)] + \
       [f"E:rec>={r},{t[:1]}" for r, t in zip(oke.min_rec, oke.term)]
hrs = list(okm.HR) + list(oke.HR); los = list(okm.lo) + list(oke.lo); his = list(okm.hi) + list(oke.hi)
cols = ["#2c3e50"] * len(okm) + ["#8e44ad"] * len(oke)
y = np.arange(len(labs))[::-1]
ax[1].errorbar(hrs, y, xerr=[np.array(hrs) - np.array(los), np.array(his) - np.array(hrs)],
               fmt="o", ecolor="#bdc3c7", capsize=2, lw=1, ms=4, linestyle="none")
ax[1].scatter(hrs, y, c=cols, s=22, zorder=3)
ax[1].axvline(1.0, ls="--", color="#c0392b", lw=1)
ax[1].set_yticks(y); ax[1].set_yticklabels(labs, fontsize=7)
ax[1].set_xlabel("HR per 24h (95% CI)"); ax[1].set_xlim(0.985, 1.015)
ax[1].set_title("Tier1 multicenter: MIMIC(navy)+eICU(purple)", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "l1_multicenter.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  saved: {FIGDIR/'l1_multicenter.png'}")


## 2. Primary analysis: early TMP surge (Tier 2)


In [ ]:
_av = lambda lst: [x for x in lst if x in cmeas.columns and not cmeas[x].isna().all()]
DEMO_C_u, SEV_C_u, LAB_C_u, TX_C_u = map(_av, [DEMO_C, SEV_C, LAB_C, TX_C])
DEMO_B_u, SEV_B_u, TX_B_u          = map(_av, [DEMO_B, SEV_B, TX_B])

MAIN_ONSET = surge_onset(tmp, *MAIN)
FULL_ONSET = surge_onset(tmp, MAIN[0], MAIN[1], 10**9)
_nexp = int(pd.Series(MAIN_ONSET).reindex(cmeas["stay_id"]).notna().sum())
print(f"MAIN surge occurred: {_nexp}/{len(cmeas)} ({_nexp/len(cmeas)*100:.1f}%)")

print("\n" + "=" * 72)
print("  surge (MAIN) adjustment hierarchy M1-M5 (28d, cmeas=%d)" % len(cmeas)); print("=" * 72)
r_m1 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, [], [])
r_m2 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, DEMO_C_u, DEMO_B_u)
r_m3 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, DEMO_C_u + SEV_C_u, DEMO_B_u + SEV_B_u)
r_m4 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, DEMO_C_u + SEV_C_u + LAB_C_u, DEMO_B_u + SEV_B_u)
r_m5_full = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET,
                          DEMO_C_u + SEV_C_u + LAB_C_u + TX_C_u,
                          DEMO_B_u + SEV_B_u + TX_B_u, full_return=True)
r_main, pooltab, CP_MAIN = r_m5_full
line("M1 surge alone",      r_m1)
line("M2 +demographics",         r_m2)
line("M3 +severity",         r_m3)
line("M4 +labs",           r_m4)
line("M5 +treatment/circuit (full)", r_main, extra="★r_main")

print("\n  [M5 covariate HR (Rubin pooled) - Table 2]")
for _, row in pooltab.iterrows():
    print(f"    {row['var']:<18} HR={row.HR:.3f} [{row.lo:.3f},{row.hi:.3f}] "
          f"p={row.p:.3f}{'*' if row.p < 0.05 else ''}")

print("\n" + "=" * 72); print("  [Bayes factor] surge 28d association (BIC approximation, event penalty)"); print("=" * 72)
_adj_M5 = [c_ for c_ in (DEMO_C_u + SEV_C_u + LAB_C_u + TX_C_u + DEMO_B_u + SEV_B_u + TX_B_u)]
bf_surge = bayes_factor_bic(CP_MAIN, "surge", _adj_M5)
line_bf("surge (M5 adjusted)", bf_surge)
print(f"  interpretation: BF10={bf_surge['BF10']:.2f} -> "
      f"{'favors H1 (association), ' + bf_surge['strength'] if bf_surge['BF10']>1 else 'favors H0 (no effect), ' + bf_surge['strength']}")
print(f"  Only n={_nexp} surge-exposed, so the surge coefficient carries little information.")
print(f"  BF in the 1-3 (weak) range means the evidence is inconclusive given the data -> hypothesis-generating.")
print(f"  (d={bf_surge['d']:.0f} is the total 28d events in the model, not surge events)")

print("\n" + "=" * 72); print("  [penalty sensitivity] M5 surge HR by penalizer"); print("=" * 72)
pen_tab = []
for pen in [0.0, 0.01, 0.1, 0.5]:
    rp = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET,
                       DEMO_C_u + SEV_C_u + LAB_C_u + TX_C_u,
                       DEMO_B_u + SEV_B_u + TX_B_u, penalizer=pen)
    pen_tab.append((pen, rp[0], rp[1], rp[2], rp[3]))
    print(f"    penalizer={pen:<5} HR={rp[0]:.3f} [{rp[1]:.3f},{rp[2]:.3f}] "
          f"p={rp[3]:.3f}{'*' if rp[3]<0.05 else ''}")
print(f"  Primary analysis penalizer={PENALIZER}. If HR is larger at penalizer=0 (unpenalized), "
      f"the primary estimate is conservative.")

print("\n" + "=" * 72); print("  [effect size] 28d mortality by surge status"); print("=" * 72)
_cm = cmeas.copy()
_cm["surge_flag"] = _cm["stay_id"].map(lambda s: 1 if pd.notna(MAIN_ONSET.get(s, np.nan)) else 0)
rd, r1, r0 = risk_diff(_cm, "surge_flag", "event28")
print(f"    surge(+) n={int(_cm['surge_flag'].sum())}: 28d mortality {r1:.1f}%")
print(f"    surge(-) n={int((_cm['surge_flag']==0).sum())}: 28d mortality {r0:.1f}%")
print(f"    observed risk difference = {rd:+.1f}pp (unadjusted, descriptive). Adjusted HR (M5)={r_main[0]:.2f}")
print("=" * 72)

_steps = [("M1 crude", r_m1), ("M2 +demo", r_m2), ("M3 +severity", r_m3),
          ("M4 +labs", r_m4), ("M5 +tx/circuit", r_main)]
_lab = [s for s, _ in _steps]; _hr = [r[0] for _, r in _steps]
_lo = [r[1] for _, r in _steps]; _hi = [r[2] for _, r in _steps]
forest_plot(_lab, _hr, _lo, _hi, "surge HR by adjustment (M1-M5)", "l2_m1m5_forest.png")

_name = {"surge": "surge(MAIN)", "anchor_age": "age", "weight_kg": "weight",
         "sofa_total": "SOFA", "map_value": "MAP", "platelet": "platelet",
         "hemoglobin": "Hb", "lactate": "lactate", "inr": "INR", "aptt": "aPTT",
         "bilirubin": "bilirubin", "blood_flow": "blood flow", "male": "male",
         "vaso_use": "vasopressor", "mech_vent": "mech vent",
         "v3_systemic_hep": "systemic hep", "v3_prophylaxis": "prophylaxis hep"}
_pt = pooltab.copy(); _pt["lab"] = _pt["var"].map(lambda v: _name.get(v, v))
_pt = _pt.sort_values("HR")
forest_plot(_pt["lab"].tolist(), _pt["HR"].tolist(), _pt["lo"].tolist(), _pt["hi"].tolist(),
            "M5 full model: covariate HRs", "l2_m5_covariates.png", figsize=(7, 6.5))
print(f"  saved: {FIGDIR/'l2_m1m5_forest.png'}, {FIGDIR/'l2_m5_covariates.png'}")

## 3. Sensitivity and robustness


In [ ]:
print(f"grid fit ({len(BASELINES)}x{len(RISES)}x{len(WINDOWS)}="
      f"{len(BASELINES)*len(RISES)*len(WINDOWS)} combinations x {M_IMP} MICE sets), MAIN={MAIN}")
results = []
for bm in BASELINES:
    for win in WINDOWS:
        for rise in RISES:
            om = surge_onset(tmp, bm, rise, win)
            ne = int(pd.Series(om).reindex(cmeas["stay_id"]).notna().sum()) if len(om) else 0
            res = fit_surge_cox(cmeas, XC_MEAS, om, CONT_M, BIN_M)
            hr, lo, hi, p = (res[0], res[1], res[2], res[3]) if res else (np.nan,) * 4
            results.append([bm, rise, win, hr, lo, hi, p, ne])
resdf = pd.DataFrame(results, columns=["baseline", "rise", "window", "HR", "lo", "hi", "p", "n_surge"])

_chk = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == MAIN[1]) &
             (resdf.window == MAIN[2])].iloc[0]
print(f"  [check] grid MAIN HR={_chk.HR:.4f} p={_chk.p:.4f}  vs  "
      f"r_main HR={r_main[0]:.4f} p={r_main[3]:.4f} -> diff {abs(_chk.HR-r_main[0]):.6f}")
assert abs(_chk.HR - r_main[0]) < 1e-6, "grid MAIN != r_main: check the fit_surge_cox/XC_MEAS path"

resdf["q_bh"] = np.nan
for bm in BASELINES:
    mask = (resdf["baseline"] == bm) & resdf["p"].notna()
    if mask.sum() > 0:
        resdf.loc[mask, "q_bh"] = bh_fdr(resdf.loc[mask, "p"].values)

print("\n" + "=" * 86)
print(f"  surge grid, 50 combinations, full TVC (28d)  |  MAIN={MAIN[0]}/{MAIN[1]}/{MAIN[2]} = r_main")
print("=" * 86)
print(f"  {'base':<8}{'rise':>5}{'win':>5}{'n':>7}{'HR':>7}{'95%CI':>16}{'p':>9}{'q(FDR)':>9}")
for _, r in resdf.iterrows():
    tag = "  <--MAIN" if (r.baseline, r.rise, r.window) == MAIN else ""
    if pd.notna(r.HR):
        ps = "*" if r.p < 0.05 else ""; qs = "+" if r.q_bh < 0.05 else ""
        print(f"  {r.baseline:<8}{int(r.rise):>5}{int(r.window):>5}{int(r.n_surge):>7}"
              f"{r.HR:>7.2f}  [{r.lo:.2f},{r.hi:.2f}]{r.p:>8.3f}{ps}{r.q_bh:>8.3f}{qs}{tag}")
    else:
        print(f"  {r.baseline:<8}{int(r.rise):>5}{int(r.window):>5}{int(r.n_surge):>7}    na{tag}")

for bm in BASELINES:
    ok = resdf[resdf.baseline == bm].dropna(subset=["HR"])
    tag = "MAIN(median)" if bm == MAIN[0] else f"Supple({bm})"
    print(f"\n  [{tag}] {len(ok)} combinations")
    print(f"    raw p<0.05 & HR>1: {len(ok[(ok.p<0.05)&(ok.HR>1)])}/{len(ok)}")
    print(f"    FDR q<0.05 & HR>1: {len(ok[(ok.q_bh<0.05)&(ok.HR>1)])}/{len(ok)}")
    print(f"    median HR {ok.HR.median():.2f}, range [{ok.HR.min():.2f},{ok.HR.max():.2f}], "
          f"HR>1 {(ok.HR>1).mean()*100:.0f}%")
_mr = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == MAIN[1]) &
            (resdf.window == MAIN[2])].iloc[0]
print(f"\n  ★ MAIN: HR={_mr.HR:.3f}, p={_mr.p:.3f}, q={_mr.q_bh:.3f}")
print("  (* raw p<0.05, + FDR q<0.05; FDR computed within each 25-combination baseline pool)")
print("=" * 86)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, bm in zip(axes, BASELINES):
    piv = resdf[resdf.baseline == bm].pivot(index="rise", columns="window", values="HR")
    im = ax.imshow(piv.values, cmap="RdBu_r", vmin=0.7, vmax=1.7, aspect="auto", origin="lower")
    ax.set_xticks(range(len(WINDOWS))); ax.set_xticklabels(WINDOWS)
    ax.set_yticks(range(len(RISES)));   ax.set_yticklabels(RISES)
    ax.set_xlabel("window (h)"); ax.set_ylabel("rise (mmHg)")
    ax.set_title(f"{'MAIN' if bm == MAIN[0] else 'Suppl'} baseline={bm}", fontsize=10, loc="left")
    for i, rise in enumerate(RISES):
        for j, win in enumerate(WINDOWS):
            v = piv.loc[rise, win]
            if pd.notna(v):
                star = "*" if (resdf[(resdf.baseline == bm) & (resdf.rise == rise) &
                                     (resdf.window == win)].p.iloc[0] < 0.05) else ""
                ax.text(j, i, f"{v:.2f}{star}", ha="center", va="center", fontsize=7,
                        color="white" if (v < 0.85 or v > 1.5) else "black")
    if bm == MAIN[0]:
        ax.add_patch(plt.Rectangle((WINDOWS.index(MAIN[2]) - 0.5, RISES.index(MAIN[1]) - 0.5),
                                   1, 1, fill=False, edgecolor="lime", lw=2.5))
fig.colorbar(im, ax=axes, label="HR", fraction=0.025)
fig.suptitle("surge HR across 50 definitions (* raw p<0.05; green box=MAIN)", fontsize=11)
plt.savefig(FIGDIR / "l2_grid_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  saved: {FIGDIR/'l2_grid_heatmap.png'}")

In [ ]:
print("=" * 72); print(f"  MAIN sensitivity (cmeas={len(cmeas)}, MAIN={MAIN})"); print("=" * 72)
_sens = [("MAIN (M5)", r_main)]

def _fit_sub(sub_mask, onset_map, cont, binc, end_col="end28_h", ev_col="event28", penalizer=PENALIZER):
    """Fit a subgroup: subset cmeas and XC_MEAS by the same boolean mask, then fit_surge_cox."""
    sub = cmeas[sub_mask].reset_index(drop=True)
    XC_sub = [Xc[sub_mask].reset_index(drop=True) for Xc in XC_MEAS]
    return fit_surge_cox(sub, XC_sub, onset_map, cont, binc, end_col, ev_col, penalizer=penalizer)

def _n_surge_in(sub_mask, onset_map):
    return int(pd.Series(onset_map).reindex(cmeas[sub_mask]["stay_id"]).notna().sum())

print("\n[1a] sepsis (strict: suspected infection & SOFA>=2, Sepsis-3)")
_m_strict = (cmeas["sepsis3"] == 1).values
_ns = _n_surge_in(_m_strict, MAIN_ONSET)
r_sepsis_s = _fit_sub(_m_strict, MAIN_ONSET, CONT_M, BIN_M)
line(f"sepsis strict (N={int(_m_strict.sum())}, ev={int(cmeas[_m_strict]['event28'].sum())}, surge={_ns})",
     r_sepsis_s)
if r_sepsis_s: _sens.append(("sepsis strict(590)", r_sepsis_s))

print("\n[1b] sepsis (broad: suspected infection & (SOFA>=2 | SOFA missing))")
_sofa_num = pd.to_numeric(cmeas["sofa_total"], errors="coerce")
_m_broad = ((cmeas["infection_susp"] == 1) & ((_sofa_num >= 2) | _sofa_num.isna())).values
_nb = _n_surge_in(_m_broad, MAIN_ONSET)
r_sepsis_b = _fit_sub(_m_broad, MAIN_ONSET, CONT_M, BIN_M)
line(f"sepsis broad (N={int(_m_broad.sum())}, ev={int(cmeas[_m_broad]['event28'].sum())}, surge={_nb})",
     r_sepsis_b)
if r_sepsis_b: _sens.append(("sepsis broad(836)", r_sepsis_b))

print("\n[2] complete-case (no missing CONT_M, no MICE)")
_cc_mask = cmeas[CONT_M].notna().all(axis=1).values
_subc = cmeas[_cc_mask].reset_index(drop=True)
_cp = build(_subc, _subc[CONT_M], MAIN_ONSET, CONT_M, BIN_M)
_fit = ["surge"] + [cc for cc in (CONT_M + BIN_M) if _cp[cc].nunique() > 1]
_m = CoxTimeVaryingFitter(penalizer=PENALIZER)
_m.fit(_cp[["id", "start", "stop", "event"] + _fit], id_col="id", start_col="start",
       stop_col="stop", event_col="event", show_progress=False)
_s = _m.summary.loc["surge"]
r_complete = (np.exp(_s["coef"]), np.exp(_s["coef lower 95%"]),
              np.exp(_s["coef upper 95%"]), _s["p"], _s["coef"], _s["se(coef)"])
line(f"complete-case (N={len(_subc)}, ev={int(_subc['event28'].sum())})", r_complete)
if r_complete: _sens.append(("complete-case", r_complete))

print("\n[3] full-window surge (no window limit)")
_nf = int(pd.Series(FULL_ONSET).reindex(cmeas["stay_id"]).notna().sum())
r_full = fit_surge_cox(cmeas, XC_MEAS, FULL_ONSET, CONT_M, BIN_M)
line(f"full-window (n_surge≈{_nf})", r_full)
if r_full: _sens.append(("full-window", r_full))

print("\n[4] 7d mortality endpoint")
cmeas["event7"] = ((cmeas["death_h"].notna()) & (cmeas["death_h"] <= FU7)).astype(int)
_e7 = np.minimum(cmeas["death_h"].fillna(np.inf), FU7)
_cz7 = (cmeas["event7"] == 0) & cmeas["disch_h"].notna() & (cmeas["disch_h"] < _e7)
_e7 = _e7.copy(); _e7[_cz7] = cmeas["disch_h"][_cz7]; cmeas["end7_h"] = _e7
r_7d = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, CONT_M, BIN_M, end_col="end7_h", ev_col="event7")
line(f"7d (ev={int(cmeas['event7'].sum())})", r_7d)
if r_7d: _sens.append(("7d endpoint", r_7d))

print("\n[5] baseline-TMP adjustment (M5 + base_median_meas)")
XC_BASE = [Xc.assign(base_median_meas=cmeas["base_median_meas"].values) for Xc in XC_MEAS]
r_base = fit_surge_cox(cmeas, XC_BASE, MAIN_ONSET, CONT_M + ["base_median_meas"], BIN_M)
line("M5 +baseline TMP (sensitivity)", r_base, extra="borderline overadjustment -> sensitivity only")
if r_base: _sens.append(("+baseline TMP", r_base))

print("\n[6] E-value (M5)")
_hr, _lo, _hi = r_main[0], r_main[1], r_main[2]
_bound = _lo if _hr > 1 else _hi
print(f"  point estimate HR={_hr:.3f} -> E-value={evalue(_hr):.2f}")
print(f"  CI bound {_bound:.3f} -> E-value={evalue(_bound):.2f}")
print("  (an unmeasured confounder would need at least this RR with both exposure and outcome to explain away the association)")

print("\n[7] proportional-hazards test (surge x log(t))")
CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C; BIN_M5 = DEMO_B + SEV_B + TX_B
_avc = lambda L: [x for x in L if x in cmeas.columns and not cmeas[x].isna().all()]
CONT_M5 = _avc(CONT_M5); BIN_M5 = _avc(BIN_M5)
ph_est, ph_var = [], []
for mi in range(M_IMP):
    Xc = XC_MEAS[mi][CONT_M5]
    cp = build_ph(cmeas, Xc, MAIN_ONSET, CONT_M5, BIN_M5)
    fit = ["surge", "surge_logt"] + [cc for cc in (CONT_M5 + BIN_M5) if cp[cc].nunique() > 1]
    m = CoxTimeVaryingFitter(penalizer=PENALIZER)
    m.fit(cp[["id", "start", "stop", "event"] + fit], id_col="id", start_col="start",
          stop_col="stop", event_col="event", show_progress=False)
    ph_est.append(m.summary.loc["surge_logt", "coef"])
    ph_var.append(m.summary.loc["surge_logt", "se(coef)"] ** 2)
_, _, _, ph_p, ph_Q, ph_se = pool_hr(ph_est, ph_var)
print(f"  surge x log(t) coef={ph_Q:.3f}, p={ph_p:.3f}{'*' if ph_p < 0.05 else ''}")
print(f"  -> not significant = PH holds (surge effect constant over follow-up); significant = time-varying")
print("=" * 72)

_lab = [s for s, _ in _sens]; _hr = [r[0] for _, r in _sens]
_lo = [r[1] for _, r in _sens]; _hi = [r[2] for _, r in _sens]
forest_plot(_lab, _hr, _lo, _hi, "surge HR: MAIN & sensitivity analyses",
            "l2_sensitivity_forest.png", figsize=(7, 0.5 * len(_sens) + 1.2))
print(f"  saved: {FIGDIR/'l2_sensitivity_forest.png'}")

In [ ]:
print("=" * 72); print("  baseline-TMP adjustment: surge vs base_median_meas HR"); print("=" * 72)

CONT_BASE = CONT_M + ["base_median_meas"]
r_base_full, base_pooltab, _ = fit_surge_cox(cmeas, XC_BASE, MAIN_ONSET,
                                             CONT_BASE, BIN_M, full_return=True)

print("\n  [surge HR comparison]")
print(f"    M5 (baseline unadjusted, primary)  HR={r_main[0]:.3f} "
      f"[{r_main[1]:.3f},{r_main[2]:.3f}] p={r_main[3]:.3f}")
print(f"    M5 + baseline-TMP adjusted         HR={r_base_full[0]:.3f} "
      f"[{r_base_full[1]:.3f},{r_base_full[2]:.3f}] p={r_base_full[3]:.3f}")

_brow = base_pooltab[base_pooltab["var"] == "base_median_meas"].iloc[0]
print(f"\n  [HR of baseline TMP itself, same model]")
print(f"    base_median_meas (per 1 mmHg)      HR={_brow.HR:.4f} "
      f"[{_brow.lo:.4f},{_brow.hi:.4f}] p={_brow.p:.3f}{'*' if _brow.p < 0.05 else ' (ns)'}")
print(f"    base_median_meas (per 10 mmHg)     HR={np.exp(np.log(_brow.HR)*10):.3f}")

print(f"\n  [interpretation]")
print(f"    surge persists (and strengthens) after adjusting for the baseline level: {r_main[0]:.2f} -> {r_base_full[0]:.2f}")
print(f"    -> the surge-mortality association is not explained by the baseline level")
print(f"    base_median_meas itself is {'significant' if _brow.p<0.05 else 'ns'} "
      f"-> the signal is the early rise, not a high absolute level")
print(f"    [borderline overadjustment: base_median is the surge reference point, its coefficient is not interpreted]")
print("=" * 72)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
_l = ["M5 (primary)", "M5 + baseline TMP"]
_h = [r_main[0], r_base_full[0]]; _lo = [r_main[1], r_base_full[1]]; _hi = [r_main[2], r_base_full[2]]
y = [1, 0]
ax[0].errorbar(_h, y, xerr=[[a - b for a, b in zip(_h, _lo)], [b - a for a, b in zip(_h, _hi)]],
               fmt="o", color="#2c3e50", ecolor="#7f8c8d", capsize=4, ms=7)
ax[0].axvline(1.0, ls="--", color="#c0392b", lw=1)
ax[0].set_yticks(y); ax[0].set_yticklabels(_l, fontsize=10); ax[0].set_ylim(-0.5, 1.5)
ax[0].set_xlabel("surge HR (95% CI)")
ax[0].set_title("surge: before/after baseline-TMP adj.", fontsize=10, loc="left")

_sf = cmeas["stay_id"].map(lambda s: 1 if s in MAIN_ONSET else 0)
b0 = cmeas[_sf == 0]["base_median_meas"].dropna()
b1 = cmeas[_sf == 1]["base_median_meas"].dropna()
ax[1].hist([b0, b1], bins=30, label=[f"no surge (n={len(b0)})", f"surge (n={len(b1)})"],
           color=["#2c3e50", "#c0392b"], density=True)
ax[1].set_xlabel("baseline TMP (mmHg)"); ax[1].set_ylabel("density"); ax[1].legend(fontsize=8)
ax[1].set_title(f"baseline TMP: HR={_brow.HR:.3f}"
                f"{' (ns)' if _brow.p >= 0.05 else ''}", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "l2_baseline_tmp.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  saved: {FIGDIR/'l2_baseline_tmp.png'}")

In [ ]:
_st = tmp.groupby("stay_id").apply(stats_12h)
_st = pd.DataFrame([x for x in _st if x is not None],
                   index=[s for s, x in _st.items() if x is not None]) \
        .reset_index().rename(columns={"index": "stay_id"})
clm = c.drop(columns=[v for v in STAT_VARS if v in c.columns], errors="ignore") \
       .merge(_st, on="stay_id", how="left")
clm["died_28d"] = clm["event28"]
print(f"12h-window statistics computed: {int(clm['tmp_max'].notna().sum())} "
      f"(slope computed for {int(clm['tmp_slope'].notna().sum())})")
print(f"  [B3] slope/time2max use the first measurement as base, unlike surge MAIN (median baseline).")
print(f"       These 7 statistics are a descriptive summary, independent of surge. All-null implies threshold-crossing is the signal.")

base_lm = clm[clm["tmp_max"].notna()].copy()
lm = base_lm[(base_lm["death_h"].isna()) | (base_lm["death_h"] > LM_H)].copy()
lm = lm[(lm["disch_h"].isna()) | (lm["disch_h"] > LM_H)].copy()
_end = np.minimum(lm["death_h"].fillna(np.inf), FU_H)
_cz = (lm["died_28d"] == 0) & lm["disch_h"].notna() & (lm["disch_h"] < _end)
_end = _end.copy(); _end[_cz] = lm["disch_h"][_cz]
lm["fu_h"] = _end - LM_H
lm["event"] = ((lm["died_28d"] == 1) & (lm["death_h"] <= FU_H)).astype(int)
lm = lm[lm["fu_h"] > 0].reset_index(drop=True)
print(f"  12h landmark survivors: {len(lm)}, subsequent 28d deaths: {int(lm['event'].sum())}")

CONT_L = [x for x in CONT_M if x in lm.columns and not lm[x].isna().all()]
BIN_L = [x for x in BIN_M if x in lm.columns]
for x in BIN_L: lm[x] = pd.to_numeric(lm[x], errors="coerce").fillna(0).astype(int)

XC_LM = mice_impute(lm, CONT_L, "event", "fu_h")

def run_stat(var, adjust):
    """HR per 1 SD. crude = single CoxPH; full = Rubin-pooled over XC_LM (no re-imputation)."""
    mask = lm[var].notna()
    d = lm[mask].reset_index(drop=True)
    if len(d) < 20 or d["event"].sum() < 5: return None, len(d), int(d["event"].sum())
    z = (d[var] - d[var].mean()) / d[var].std()
    if not adjust:
        dd = pd.DataFrame({"fu_h": d["fu_h"], "event": d["event"], "_z": z})
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(dd, duration_col="fu_h", event_col="event")
        s = cph.summary.loc["_z"]
        return (np.exp(s["coef"]), np.exp(s["coef lower 95%"]),
                np.exp(s["coef upper 95%"]), s["p"]), len(d), int(d["event"].sum())
    est, var_ = [], []
    for mi in range(M_IMP):
        Xc = XC_LM[mi][mask.values].reset_index(drop=True)
        dd = pd.DataFrame({"fu_h": d["fu_h"].values, "event": d["event"].values, "_z": z.values})
        for cc in CONT_L: dd[cc] = Xc[cc].values
        for cc in BIN_L:  dd[cc] = d[cc].values
        fit = ["_z"] + [cc for cc in CONT_L + BIN_L if dd[cc].nunique() > 1]
        cph = CoxPHFitter(penalizer=PENALIZER)
        cph.fit(dd[["fu_h", "event"] + fit], duration_col="fu_h", event_col="event")
        est.append(cph.summary.loc["_z", "coef"]); var_.append(cph.summary.loc["_z", "se(coef)"]**2)
    hr, lo, hi, p, _, _ = pool_hr(est, var_)
    return (hr, lo, hi, p), len(d), int(d["event"].sum())

print("\n" + "=" * 80)
print("  Early 12h TMP statistics -> 28d mortality (landmark, HR per 1 SD)")
print("  Absolute level (min/mean/median/max) + change (range/time2max/slope). All-null implies threshold-crossing is the signal")
print("=" * 80)
print(f"  {'statistic':<16}{'model':<7}{'HR':>8}{'95%CI':>18}{'p':>9}{'N':>7}{'ev':>5}")
LABEL = {"tmp_min": "min", "tmp_mean": "mean", "tmp_median": "median", "tmp_max": "max",
         "tmp_range": "range(max-min)", "tmp_time2max": "time-to-max", "tmp_slope": "slope(v0 base)"}
_fres = []
for var in STAT_VARS:
    for adjust, mname in [(False, "crude"), (True, "full")]:
        res, n, ev = run_stat(var, adjust)
        if res is None:
            print(f"  {LABEL[var]:<16}{mname:<7} insufficient sample (N={n},ev={ev})"); continue
        hr, lo, hi, p = res; star = "*" if p < 0.05 else ""
        print(f"  {LABEL[var]:<16}{mname:<7}{hr:>8.3f}  [{lo:.3f},{hi:.3f}]{p:>8.3f}{star}{n:>7}{ev:>5}")
        if adjust: _fres.append((LABEL[var], hr, lo, hi))
print("=" * 80)
_nsig = sum(1 for _, h, l, hh in _fres if not (l <= 1 <= hh))
print(f"  significant (CI excludes 1) in the full model: {_nsig}/7")
print("  -> neither absolute-level nor change summaries relate to mortality => the signal is specific to the early threshold-crossing event")

print("\n  [B3, additional] median-baseline slope (baseline matched to surge MAIN)")
_slope_med = []
for sid, g in tmp.groupby("stay_id"):
    w = g[(g["h"] >= 0) & (g["h"] <= LM_H)].sort_values("h")
    if len(w) < 2: continue
    b = w[w["h"] <= 3]["valuenum"]
    base = b.median() if len(b) else w["valuenum"].iloc[0]
    vmax = w["valuenum"].max(); tmax = w.loc[w["valuenum"].idxmax(), "h"]
    _slope_med.append((sid, (vmax - base) / tmax if tmax > 0 else np.nan))
_sm = pd.DataFrame(_slope_med, columns=["stay_id", "tmp_slope_med"])
lm = lm.merge(_sm, on="stay_id", how="left")
res_sm, n_sm, ev_sm = run_stat("tmp_slope_med", True)
if res_sm:
    print(f"    slope(median base) full  HR={res_sm[0]:.3f} "
          f"[{res_sm[1]:.3f},{res_sm[2]:.3f}] p={res_sm[3]:.3f} (N={n_sm}, ev={ev_sm})")
    print(f"    -> null, same as the first-measurement-base slope: "
          f"smooth change is not a signal, regardless of the baseline definition")

forest_plot([l for l, _, _, _ in _fres], [h for _, h, _, _ in _fres],
            [l for _, _, l, _ in _fres], [hh for _, _, _, hh in _fres],
            "12h TMP summary stats -> 28d (full, 1 SD)", "l2_landmark_forest.png", figsize=(7, 4.2))
print(f"  saved: {FIGDIR/'l2_landmark_forest.png'}")

In [ ]:
from scipy.stats import norm as _norm

_beta = np.log(r_main[0]); _se = (np.log(r_main[2]) - np.log(r_main[1])) / (2 * 1.96)
_N0 = len(cmeas); _d0 = int(cmeas["event28"].sum()); _nexp0 = _nexp
print("=" * 76); print("  Information diagnostics (conditional power, BF design analysis, prior sensitivity)"); print("=" * 76)
print(f"  observed (M5): HR={r_main[0]:.3f}, logHR beta={_beta:.4f}, SE={_se:.4f}, "
      f"z={_beta/_se:.2f}, N={_N0}, event={_d0}, surge={_nexp0}")

print("\n" + "-" * 76)
print("  [A] conditional power (assumed effect size x sample-size expansion; SE ~ 1/sqrt(N))")
print("-" * 76)
HR_SCEN = [1.30, r_main[0], 1.60]
N_SCEN = [_N0, 1500, 3000, 5000, 8000]
powtab = {}
print(f"  {'N':>6}{'surge≈':>8}" + "".join(f"{'HR='+format(h,'.2f'):>12}" for h in HR_SCEN))
for N in N_SCEN:
    k = N / _N0; se_k = _se / np.sqrt(k); row = []
    for h in HR_SCEN:
        b = np.log(h)
        power = _norm.cdf(b/se_k - 1.96) + _norm.cdf(-b/se_k - 1.96)
        row.append(power)
    powtab[N] = row
    print(f"  {N:>6}{int(round(_nexp0*k)):>8}" + "".join(f"{p*100:>11.0f}%" for p in row))
print(f"  -> at the observed effect (HR={r_main[0]:.2f}), power at current N={powtab[_N0][1]*100:.0f}%, "
      f"at N~3000 it would be {powtab[3000][1]*100:.0f}%. The signal is consistent; only information is limited.")

print("\n" + "-" * 76)
print("  [B] BF design analysis (deltaBIC=z^2-ln(d), events assumed proportional to N)")
print("-" * 76)
_z0 = _beta/_se; _bf10_approx0 = np.exp((_z0**2 - np.log(_d0))/2)
print(f"  [check] approx BF10 at N={_N0} = {_bf10_approx0:.2f}  vs  observed BF10={bf_surge['BF10']:.2f} "
      f"(diff {abs(_bf10_approx0-bf_surge['BF10']):.2f})")
print(f"  {'N':>6}{'z':>7}{'d':>7}{'BF10':>12}{'evidence(H1)':>14}")
def _bf_strength(bf10):
    b = bf10 if bf10 >= 1 else 1/bf10
    s = "very strong" if b > 150 else "strong" if b > 20 else "positive" if b > 3 else "weak"
    return ("H1 " if bf10 >= 1 else "H0 ") + s
cross_N = None
for N in N_SCEN:
    k = N / _N0; se_k = _se / np.sqrt(k); z = _beta/se_k; d = _d0 * k
    bf10 = np.exp((z**2 - np.log(d)) / 2)
    if cross_N is None and bf10 > 3: cross_N = N
    print(f"  {N:>6}{z:>7.2f}{d:>7.0f}{bf10:>12.2f}{_bf_strength(bf10):>14}")
print(f"  -> at the observed effect, BF10 crosses 'positive (H1)' (>3) near N={cross_N}. "
      f"the current inconclusiveness reflects limited information, not absence of effect.")

print("\n" + "-" * 76)
print("  [C] prior sensitivity (N(0, tau^2) prior on log-HR; Savage-Dickey approximation)")
print("-" * 76)
print(f"  {'prior tau (log-HR SD)':>22}{'implies (HR 1SD)':>16}{'BF01':>9}{'BF10':>9}{'evidence':>14}")
for tau in [0.25, 0.50, 1.00, 2.00]:
    post0 = _norm.pdf(0, loc=_beta, scale=_se)
    prior0 = _norm.pdf(0, loc=0, scale=tau)
    bf01 = post0 / prior0; bf10 = 1/bf01
    print(f"  {tau:>22.2f}{('HR~'+format(np.exp(tau),'.2f')):>16}"
          f"{bf01:>9.2f}{bf10:>9.2f}{_bf_strength(bf10):>14}")
print(f"  -> the BF direction depends on the prior (small tau favors H1, large tau favors H0),")
print(f"     but |BF| stays in the 1-5 'weak evidence' range throughout - neither strongly H0 nor H1.")
print(f"     this inconclusiveness is itself the quantitative case for a hypothesis-generating framing.")
print(f"     (the multivariable BIC above leans weakly H0 via the event penalty; the univariate Savage-Dickey")
print(f"      is prior-dependent - both agree only on 'weak evidence')")
print("=" * 76)

design_N = N_SCEN
design_power = [powtab[N][1] * 100 for N in N_SCEN]
design_bf10 = [np.exp((_beta/(_se/np.sqrt(N/_N0)))**2/2 - np.log(_d0*N/_N0)/2) for N in N_SCEN]
design_N0 = _N0

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
_cols = ["#7f8c8d", "#c0392b", "#2c3e50"]
for h, col in zip(HR_SCEN, _cols):
    ys = [powtab[N][HR_SCEN.index(h)] * 100 for N in N_SCEN]
    ax[0].plot(N_SCEN, ys, "o-", color=col, label=f"HR={h:.2f}" + (" (obs)" if abs(h-r_main[0])<1e-6 else ""))
ax[0].axhline(80, ls="--", color="gray", lw=1); ax[0].axvline(_N0, ls=":", color="#c0392b", lw=1)
ax[0].set_xlabel("total N (surge rate fixed 5.3%)"); ax[0].set_ylabel("power (%)")
ax[0].set_title("Conditional power vs sample size", fontsize=11, loc="left")
ax[0].legend(fontsize=9); ax[0].set_ylim(0, 105)
bf_ys = []
for N in N_SCEN:
    k = N/_N0; z = _beta/(_se/np.sqrt(k)); bf_ys.append(np.exp((z**2 - np.log(_d0*k))/2))
ax[1].plot(N_SCEN, bf_ys, "o-", color="#16a085")
ax[1].axhline(3, ls="--", color="gray", lw=1); ax[1].axhline(1, ls="-", color="black", lw=0.6)
ax[1].axvline(_N0, ls=":", color="#c0392b", lw=1)
ax[1].set_yscale("log"); ax[1].set_xlabel("total N"); ax[1].set_ylabel("BF10 (log scale)")
ax[1].set_title("BF design analysis (obs effect held)", fontsize=11, loc="left")
ax[1].text(N_SCEN[-1], 3.3, "positive (H1)", ha="right", fontsize=8, color="gray")
plt.tight_layout(); plt.savefig(FIGDIR / "l2_power_bf_design.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  saved: {FIGDIR/'l2_power_bf_design.png'}")

In [ ]:
RISE_E = MAIN[1]; DET_E = MAIN[2]
COV_E = [c for c in (CONT_M + BIN_M) if c not in ["base_median_meas"]]

d0 = pd.read_parquet(COHORT_PARQUET)
for col in ["t0", "crrt_end", "last_dischtime", "death_time_final"]:
    if col in d0.columns: d0[col] = pd.to_datetime(d0[col], errors="coerce")
d0 = d0[d0["base_median_meas"].notna()].copy().reset_index(drop=True)

_death_h = (d0["death_time_final"] - d0["t0"]).dt.total_seconds() / 3600
_disch_h = (d0["last_dischtime"] - d0["t0"]).dt.total_seconds() / 3600
d0["death_h"] = _death_h
d0["event"] = ((_death_h.notna()) & (_death_h <= FU_H)).astype(int)
d0["end_h"] = np.minimum(_death_h.fillna(np.inf), FU_H)
_cens = (d0["event"] == 0) & _disch_h.notna() & (_disch_h < d0["end_h"])
d0.loc[_cens, "end_h"] = _disch_h[_cens]
d0 = d0[d0["end_h"] > 0].reset_index(drop=True)

_base = d0["base_median_meas"]; _peak = d0[f"peak_{DET_E}h_meas"]; _ptime = d0[f"peak_{DET_E}h_time_meas"]
d0["surge_h"] = np.where(((_peak - _base >= RISE_E) & _ptime.notna() & (_ptime <= DET_E)),
                         _ptime, np.nan)
d0["term_h"] = (d0["crrt_end"] - d0["t0"]).dt.total_seconds() / 3600
d0["term_h"] = np.minimum(d0["term_h"], d0["end_h"])
d0["btmp"] = _base

print("=" * 72); print(f"  surge vs termination contrast (N={len(d0)}, 28d deaths={int(d0['event'].sum())})")
print("=" * 72)
print(f"  surge(≤{DET_E}h, Δ≥{RISE_E}): {int(d0['surge_h'].notna().sum())}")
print(f"  term_h median={d0['term_h'].median():.1f}h, <12h={(d0['term_h']<12).mean()*100:.0f}% "
      f"<24h={(d0['term_h']<24).mean()*100:.0f}% <48h={(d0['term_h']<48).mean()*100:.0f}%")

def build_cp_grace(d, Xi, cols, expo_h_col, expo_within, grace_h, tag):
    """Start-stop builder with a grace option: deaths within grace_h of exposure are censored (removes reverse causation)."""
    rows = []
    for i in range(len(d)):
        end = d.iloc[i]["end_h"]; ev = int(d.iloc[i]["event"]); eh = d.iloc[i][expo_h_col]
        cov = {c: float(Xi.iloc[i][c]) for c in cols}
        exposed = pd.notna(eh) and (expo_within is None or eh <= expo_within) and (eh < end)
        if not exposed:
            rows.append({"id": f"{tag}_{i}", "start": 0., "stop": end, "expo": 0, "event": ev, **cov})
            continue
        e = max(eh, 0.001); ev_post = ev
        if grace_h > 0 and ev == 1 and (end - e) < grace_h: ev_post = 0
        rows.append({"id": f"{tag}_{i}", "start": 0., "stop": e, "expo": 0, "event": 0, **cov})
        rows.append({"id": f"{tag}_{i}", "start": e, "stop": end, "expo": 1, "event": ev_post, **cov})
    cp = pd.DataFrame(rows); cp = cp[cp["stop"] > cp["start"]]
    drop = [c for c in cols if cp[c].nunique() <= 1]
    return cp.drop(columns=drop)

def fit_pool_grace(d, expo_h_col, expo_within, grace_h, cols, label):
    """MICE (M_IMP sets) + Rubin pooling for the exposure HR. Uses its own imputation
       (not the shared mice_impute) because grace changes the event/endpoint definition."""
    est, var = [], []
    for mi in range(M_IMP):
        imp = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=SEED + mi,
                               sample_posterior=True, initial_strategy="median")
        Xi = pd.DataFrame(imp.fit_transform(d[cols]), columns=cols, index=d.index)
        cp = build_cp_grace(d, Xi, cols, expo_h_col, expo_within, grace_h, f"m{mi}")
        if cp["expo"].sum() == 0 or cp[cp.expo == 1]["event"].sum() == 0: return None
        m = CoxTimeVaryingFitter(penalizer=PENALIZER)
        m.fit(cp, id_col="id", start_col="start", stop_col="stop", event_col="event", show_progress=False)
        est.append(m.summary.loc["expo", "coef"]); var.append(m.summary.loc["expo", "se(coef)"]**2)
    hr, lo, hi, p, _, _ = pool_hr(est, var)
    n_expo = int(((d[expo_h_col].notna()) & ((expo_within is None) | (d[expo_h_col] <= expo_within)) &
                  (d[expo_h_col] < d["end_h"])).sum())
    return hr, lo, hi, p, n_expo

_cols = [c for c in COV_E if c in d0.columns and not d0[c].isna().all()] + ["btmp"]
print(f"\n  {'exposure':<24}{'grace':>7}{'n_expo':>8}{'HR':>7}{'95%CI':>18}{'p':>9}")
surge_grace = []
for g in [0, 6, 24]:
    r = fit_pool_grace(d0, "surge_h", DET_E, g, _cols, "surge")
    if r:
        surge_grace.append((r[0], r[1], r[2]))
        print(f"  {'TMP surge(≤12h)':<24}{g:>7}{r[4]:>8}{r[0]:>7.2f}  "
              f"[{r[1]:.2f},{r[2]:.2f}]{r[3]:>8.3f}{'*' if r[3]<0.05 else ''}")
term_grace = {24: [], 48: []}
for C in [24, 48]:
    for g in [0, 6, 24]:
        r = fit_pool_grace(d0, "term_h", C, g, _cols, f"term{C}")
        if r:
            term_grace[C].append((r[0], r[1], r[2]))
            print(f"  {f'termination(<={C}h)':<24}{g:>7}{r[4]:>8}{r[0]:>7.2f}  "
                  f"[{r[1]:.2f},{r[2]:.2f}]{r[3]:>8.3f}{'*' if r[3]<0.05 else ''}")

_e = d0["end_h"].values; _eh = d0["surge_h"].values; _ev = d0["event"].values
_exposed24 = pd.notna(d0["surge_h"]) & (d0["surge_h"] <= DET_E) & (d0["surge_h"] < d0["end_h"])
_post24_death = int(((_exposed24) & (d0["event"]==1) & ((d0["end_h"] - d0["surge_h"].fillna(0)) >= 24)).sum())
_surge_ev_total = int(((_exposed24) & (d0["event"]==1)).sum())
print(f"\n  [diagnostic] of {_surge_ev_total} deaths among surge-exposed, "
      f"died after surviving 24h+ post-surge (remaining under grace24): {_post24_death}")
print(f"    few remaining events => the grace24 attenuation reflects power, not reverse causation")

print("\n  interpretation (key comparison at grace 6h):")
print("  termination: collapses from 3.72 to 1.41 at grace 6h -> the signal reflects deaths right after termination (reverse causation).")
print("  surge: holds at 1.55->1.40 at grace 6h -> surge precedes death by 6h+ (not reverse causation).")
print("    same grace 6h: termination collapses, surge holds - an asymmetry in lead time (direct evidence).")
print("  Note: grace 24h exceeds the surge time scale (onset within 12h), so it censors genuine signal too")
print("    and is excessive for surge; use it only to gauge the strength of termination's reverse causation.")
print("=" * 72)

In [ ]:
cE = pd.read_parquet(COHORT_PARQUET)
for col in ["t0", "crrt_end"]:
    if col in cE.columns: cE[col] = pd.to_datetime(cE[col], errors="coerce")
cE["stay_id"] = cE["stay_id"].astype("int64")

_tm = tmp.merge(cE[["stay_id", "t0", "crrt_end"]], on="stay_id", how="inner")
_tm["ct"] = _tm["t0"] + pd.to_timedelta(_tm["h"], unit="h")
_tm = _tm[(_tm["ct"] >= _tm["t0"]) & (_tm["ct"] <= _tm["crrt_end"])]
_tm["is_base"] = _tm["ct"] <= _tm["t0"] + pd.Timedelta(hours=3)
_b = _tm[_tm["is_base"]].groupby("stay_id")["valuenum"].median().rename("base_tmp")
_pk = _tm.groupby("stay_id")["valuenum"].max().rename("max_tmp")
_nc = _tm.groupby("stay_id")["valuenum"].size().rename("n_tmp")
rise = pd.concat([_b, _pk, _nc], axis=1); rise["max_rise"] = rise["max_tmp"] - rise["base_tmp"]
rise = rise[rise["n_tmp"] >= 2]
cA = c.merge(rise[["base_tmp", "max_tmp", "max_rise"]], on="stay_id", how="left")
has_tmp = cA["max_tmp"].notna()
print(f"pre-termination TMP metrics available: {int(has_tmp.sum())} patients")

print("\n" + "=" * 64); print("[A] pre-termination TMP metric distribution (measured TMP available)"); print("=" * 64)
sub = cA[has_tmp]; Nv = len(sub)
print(f"  N={Nv}")
print("  -- max rise above baseline --")
for thr in [50, 100]:
    n = int((sub["max_rise"] >= thr).sum()); print(f"  Δ≥{thr:>3}: {n:>4} ({n/Nv*100:.1f}%)")
print("  -- absolute max TMP --")
for thr in [250, 300]:
    n = int((sub["max_tmp"] >= thr).sum()); print(f"  absolute>={thr:>3}: {n:>4} ({n/Nv*100:.1f}%)")

_p12 = cA["peak_12h_meas"]; _bm = cA["base_median_meas"]; _pt = cA["peak_12h_time_meas"]
cA["surge_main"] = ((_p12 - _bm >= 100) & _pt.notna() & (_pt <= 12)).astype(int)
cA["abs250"] = (cA["max_tmp"] >= 250).astype(int)
ct = pd.crosstab(cA.loc[has_tmp, "surge_main"], cA.loc[has_tmp, "abs250"],
                 rownames=["surge(≤12h,Δ≥100)"], colnames=["pre-termination absolute>=250"])
print("\n  [cross-tab] early surge vs pre-termination absolute>=250:")
print(ct.to_string())
_ov = int(((cA["surge_main"] == 1) & (cA["abs250"] == 1) & has_tmp).sum())
_sn = int(((cA["surge_main"] == 1) & has_tmp).sum())
_an = int(((cA["abs250"] == 1) & has_tmp).sum())
_un = _sn + _an - _ov
print(f"\n  of {_sn} surge patients, {_ov} also reach absolute>=250 ({_ov/max(_sn,1)*100:.0f}%)")
print(f"  of {_an} absolute>=250 patients, {_ov} are surge ({_ov/max(_an,1)*100:.0f}%) "
      f"-> {(1-_ov/max(_an,1))*100:.0f}% of absolute>=250 patients are not surge")
print(f"  Jaccard overlap of the two groups: {_ov}/{_un} = {_ov/max(_un,1)*100:.0f}% (low)")
print(f"  -> early surge and late high pressure are largely distinct -> surge != filter exhaustion")

print("\n" + "=" * 64); print("[B] cumulative operating time (TVC) -> 28d mortality, by handling of the absolute>=250 group"); print("=" * 64)
n_up = int(((cA["abs250"] == 1) & has_tmp).sum())
n_not = int(((cA["abs250"] == 0) & has_tmp).sum())
print(f"  of {int(has_tmp.sum())} with measured TMP - reached high pressure (>=250): {n_up} / did not: {n_not}")

filter_tvc = {}
def _grid_sub(mask_series, label, key=None):
    m = (mask_series & (cA["end28_h"].notna()) & (cA["end28_h"] > 0)).values
    d = cA[m].reset_index(drop=True)
    if len(d) < 30 or d["event28"].sum() < 5:
        print(f"  {label:<32} insufficient sample (N={len(d)},ev={int(d['event28'].sum())})"); return
    hr, lo, hi, p, n, ev = fit_grid_cox(d, XC_C, SEG_MAIN, CONT_M, BIN_M,
                                        "end28_h", "event28", builder="mimic", scale=24,
                                        mask=m)
    if key: filter_tvc[key] = (hr, lo, hi)
    print(f"  {label:<32} dur/24h HR={hr:.3f} [{lo:.3f},{hi:.3f}] "
          f"p={p:.3f}{'*' if p<0.05 else ''} (N={n},ev={ev})")

_grid_sub(pd.Series(True, index=cA.index), "(b0) full cohort (=Tier 1)", key="b0")
_grid_sub(has_tmp & (cA["abs250"] == 0), "(b1) absolute<250, did not reach (key)", key="b1")
_grid_sub(has_tmp & (cA["abs250"] == 1), "(b2) absolute>=250, reached", key="b2")
_grid_sub(has_tmp, "(b3) all with measured TMP", key="b3")

surge_overlap = {"surge_only": _sn - _ov, "overlap": _ov, "filter_only": _an - _ov}

print("\n" + "=" * 64); print("summary:")
print("  [A] low Jaccard overlap between surge (delta>=100) and absolute>=250 -> early surge != filter exhaustion")
print("  [B] operating-time per-24h HR~1 even in the absolute<250 group (same TVC method as Tier 1) ->")
print("      the Tier 1 null is not driven by routine filter-change patients; operating time itself is unrelated to prognosis.")
print("=" * 64)

## 4. Figures


In [ ]:
WIN = 12; GRID = np.arange(0, WIN + 0.5, 0.5)
_tby = {s: g.sort_values("h") for s, g in tmp.groupby("stay_id")}

def _interp(sid):
    g = _tby.get(sid)
    if g is None: return None
    w = g[(g["h"] >= 0) & (g["h"] <= WIN)]
    if len(w) < 2: return None
    h = w["h"].values; v = w["valuenum"].values
    out = np.full(len(GRID), np.nan)
    for i, t in enumerate(GRID):
        if h[0] <= t <= h[-1]: out[i] = np.interp(t, h, v)
    return out

_sids = cmeas["stay_id"].values
_died = pd.Series(cmeas["event28"].values, index=_sids)
_surge = pd.Series([1 if s in MAIN_ONSET else 0 for s in _sids], index=_sids)
_traj = {s: _interp(s) for s in _sids}; _traj = {s: t for s, t in _traj.items() if t is not None}
g_s = [s for s in _sids if _surge.get(s) == 1 and s in _traj]
g_ns = [s for s in _sids if _surge.get(s) == 0 and s in _traj]
print(f"trajectories available: {len(_traj)}/{len(_sids)}")
print(f"  surge {len(g_s)} (deaths {int(_died.loc[g_s].sum())}, {_died.loc[g_s].mean()*100:.1f}%)")
print(f"  no-surge {len(g_ns)} (deaths {int(_died.loc[g_ns].sum())}, {_died.loc[g_ns].mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for ax, grp, title, col in [(axes[0], g_s, "Surge", "#c0392b"),
                            (axes[1], g_ns, "No surge", "#2c3e50")]:
    for s in grp: ax.plot(GRID, _traj[s], color=col, alpha=0.12, lw=0.6)
    if grp:
        arr = np.array([_traj[s] for s in grp])
        ax.plot(GRID, np.nanmean(arr, axis=0), color="black", lw=2.5, label=f"mean (n={len(grp)})")
    ax.set_title(f"{title} (0-12h trajectories)", fontsize=11, loc="left")
    ax.set_xlabel("Hours from CRRT start"); ax.axvline(3, ls=":", color="gray", lw=1)
    ax.legend(loc="upper left", fontsize=9); ax.set_xlim(0, WIN)
axes[0].set_ylabel("TMP (mmHg)")
plt.tight_layout(); plt.savefig(FIGDIR / "l2_trajectories.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  saved: {FIGDIR/'l2_trajectories.png'}")

In [ ]:
_chk = r_l1m[r_l1m["endpoint"] != "28d"]
print("in-hospital endpoint settings:", len(_chk), "(should be 6)")
print("settings with NaN:", _chk["HR"].isna().sum(), "(should be 0)")
print(_chk[["source","gap","HR","lo","hi","ev"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

mpl.rcParams.update({
    'font.size': 8, 'axes.linewidth': 0.8, 'axes.edgecolor': '#333333',
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'figure.dpi': 300, 'savefig.dpi': 300,
})
NAVY = '#2c3e50'; BRONZE = '#8c6d3f'; REDLINE = '#b03a2e'; GREY = '#cccccc'

def _logfix(ax, axis='x'):
    a = ax.xaxis if axis == 'x' else ax.yaxis
    a.set_major_formatter(mticker.FixedFormatter(['1.0', '1.5', '2.0']))
    a.set_minor_formatter(mticker.NullFormatter())
    a.set_minor_locator(mticker.NullLocator())

def _mlab(row):
    src = 'Procedure' if str(row['source']).startswith('proc') else 'Union'
    return f"{src}, {int(row['gap'])} h"
def _elab(row):
    rec = f"≥{int(row['min_rec'])} records"
    tm = 'last' if 'last' in str(row['term']) or str(row['term']).startswith('a') else '+gap'
    return f"{rec}, {tm}"

_m = r_l1m.dropna(subset=["HR"]).copy()
_m28 = _m[_m["endpoint"] == "28d"].sort_values(["source", "gap"]).reset_index(drop=True)
_mih = _m[_m["endpoint"] != "28d"].sort_values(["source", "gap"]).reset_index(drop=True)
_e = r_l1e.dropna(subset=["HR"]).reset_index(drop=True)

groups = [
    ('MIMIC-IV, 28-day mortality',
     [_mlab(r) for _, r in _m28.iterrows()], list(_m28.HR), list(_m28.lo), list(_m28.hi), NAVY),
    ('MIMIC-IV, in-hospital mortality',
     [_mlab(r) for _, r in _mih.iterrows()], list(_mih.HR), list(_mih.lo), list(_mih.hi), NAVY),
    ('eICU, in-hospital mortality',
     [_elab(r) for _, r in _e.iterrows()], list(_e.HR), list(_e.lo), list(_e.hi), BRONZE),
]

labels = []; hr = []; lo = []; hi = []; cols = []
yrows = []; headers = []; ypos = 0.0
for gname, glab, ghr, glo, ghi, gcol in groups:
    headers.append((ypos, gname)); ypos += 1.0
    for k in range(len(glab)):
        labels.append(glab[k]); hr.append(ghr[k]); lo.append(glo[k]); hi.append(ghi[k])
        cols.append(gcol); yrows.append(ypos); ypos += 1.0
    ypos += 0.6
ytop = ypos
yrows = np.array([ytop - y for y in yrows])
headers = [(ytop - hy, hn) for hy, hn in headers]
n = len(hr)

fig, ax = plt.subplots(figsize=(4.4, 5.2))
for i in range(n):
    ax.plot([lo[i], hi[i]], [yrows[i], yrows[i]], '-', color=cols[i], lw=1.1,
            solid_capstyle='round', alpha=0.9)
ax.scatter(hr, yrows, c=cols, s=18, zorder=3, edgecolors='white', linewidths=0.4)
ax.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
ax.set_xlim(0.985, 1.015); ax.set_xticks([0.99, 1.00, 1.01])
ax.set_yticks(yrows); ax.set_yticklabels(labels, fontsize=7)
ax.set_ylim(yrows.min() - 0.8, ytop + 0.5)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.spines['left'].set_visible(False); ax.tick_params(axis='y', length=0)
ax.set_xlabel('Hazard ratio per 24 h')
for hy, hn in headers:
    ax.text(-0.02, hy, hn, transform=ax.get_yaxis_transform(), ha='right', va='center',
            fontsize=7.5, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGDIR / 'Fig2.png', bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / 'Fig2.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print(f"[Fig 2] {n} settings labeled, HR {min(hr):.3f}~{max(hr):.3f}")

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

mpl.rcParams.update({
    'font.size': 8, 'axes.linewidth': 0.8, 'axes.edgecolor': '#333333',
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'figure.dpi': 300, 'savefig.dpi': 300,
})
NAVY = '#2c3e50'; BRONZE = '#8c6d3f'; REDLINE = '#b03a2e'

def _logfix(ax, axis='x'):
    a = ax.xaxis if axis == 'x' else ax.yaxis
    a.set_major_formatter(mticker.FixedFormatter(['1.0', '1.5', '2.0']))
    a.set_minor_formatter(mticker.NullFormatter()); a.set_minor_locator(mticker.NullLocator())

fig = plt.figure(figsize=(4.2, 6.8), constrained_layout=True)
gs = fig.add_gridspec(3, 1, height_ratios=[1.05, 1.35, 0.72])

axA = fig.add_subplot(gs[0])
_steps = [r_m1, r_m2, r_m3, r_m4, r_main]
hrA = [s[0] for s in _steps]; loA = [s[1] for s in _steps]; hiA = [s[2] for s in _steps]
labA = ['M1 crude', 'M2 + demographics', 'M3 + severity',
        'M4 + laboratory', 'M5 + treatment']
yA = np.arange(5)[::-1]
for i in range(5):
    axA.plot([loA[i], hiA[i]], [yA[i], yA[i]], '-', color=NAVY, lw=1.2,
             solid_capstyle='round', alpha=0.9)
axA.scatter(hrA, yA, c=NAVY, s=22, zorder=3, edgecolors='white', linewidths=0.5)
axA.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
axA.set_xscale('log'); axA.set_xlim(0.95, 2.5); axA.set_xticks([1.0, 1.5, 2.0]); _logfix(axA, 'x')
axA.set_ylim(-0.5, 4.5); axA.set_yticks(yA); axA.set_yticklabels(labA, fontsize=7.5)
for s in ['top', 'right']: axA.spines[s].set_visible(False)
axA.set_xlabel('Hazard ratio', fontsize=8)
axA.set_title('A', fontsize=10, fontweight='bold', loc='left', x=-0.02)

gsB = gs[1].subgridspec(1, 2, wspace=0.10)
cmap = LinearSegmentedColormap.from_list(
    'hr', ['#efe9da', '#d8c9a8', '#c4a978', '#a8854f', '#6b6f6e', '#2c3e50'])
hmnorm = TwoSlopeNorm(vmin=min(0.95, resdf["HR"].min()),
                      vcenter=float(resdf["HR"].median()),
                      vmax=max(1.45, resdf["HR"].max()))
_xt = [0, 1, 3, 4]; _xtl = [WINDOWS[i] for i in _xt]
for j, bm in enumerate([MAIN[0], 'first']):
    axB = fig.add_subplot(gsB[j])
    piv = (resdf[resdf.baseline == bm]
           .pivot(index="rise", columns="window", values="HR")
           .sort_index(ascending=False))
    axB.imshow(piv.values, cmap=cmap, norm=hmnorm, aspect='equal')
    axB.set_xticks(_xt); axB.set_xticklabels(_xtl, fontsize=7)
    if j == 0:
        axB.set_yticks(range(len(RISES))); axB.set_yticklabels(RISES[::-1], fontsize=7)
        axB.set_ylabel('Rise threshold (mmHg)', fontsize=7.5)
    else:
        axB.set_yticks([])
    axB.set_xlabel('Window (h)', fontsize=7.5)
    axB.set_title('Median' if j == 0 else 'First', fontsize=8, pad=2)
    for s in ['top', 'right', 'left', 'bottom']: axB.spines[s].set_visible(False)
    if j == 0:
        axB.text(-0.30, 1.18, 'B', transform=axB.transAxes, fontsize=10, fontweight='bold')

axC = fig.add_subplot(gs[2])
_r100 = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == 100) & (resdf.window == 12)].iloc[0]
_r150 = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == 150) & (resdf.window == 12)].iloc[0]
try:
    _wc = (r_whole[0], r_whole[1], r_whole[2])
except NameError:
    _rw = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == 100) &
                (resdf.window == resdf["window"].max())].iloc[0]
    _wc = (_rw["HR"], _rw["lo"], _rw["hi"])
ypos = [2, 1, 0]
hrC = [_r100["HR"], _r150["HR"], _wc[0]]
loC = [_r100["lo"], _r150["lo"], _wc[1]]
hiC = [_r100["hi"], _r150["hi"], _wc[2]]
colsC = [NAVY, NAVY, BRONZE]
labC = ['100 mmHg / 12 h', '150 mmHg / 12 h', 'Whole period']
for i in range(3):
    axC.plot([loC[i], hiC[i]], [ypos[i], ypos[i]], '-', color=colsC[i], lw=1.2,
             solid_capstyle='round', alpha=0.9)
axC.scatter(hrC, ypos, c=colsC, s=22, zorder=3, edgecolors='white', linewidths=0.5)
axC.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
axC.set_xscale('log'); axC.set_xlim(0.82, 2.6); axC.set_xticks([1.0, 1.5, 2.0]); _logfix(axC, 'x')
axC.set_ylim(-0.6, 2.6); axC.set_yticks(ypos); axC.set_yticklabels(labC, fontsize=7.5)
for s in ['top', 'right']: axC.spines[s].set_visible(False)
axC.set_xlabel('Hazard ratio', fontsize=8)
axC.set_title('C', fontsize=10, fontweight='bold', loc='left', x=-0.02)

fig.savefig(FIGDIR / 'Fig3.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'Fig3.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print(f"[Fig 3] A:{hrA[0]:.2f}->{hrA[-1]:.2f} | C:{hrC[0]:.2f}/{hrC[1]:.2f}/{hrC[2]:.2f}")

In [ ]:
from scipy.stats import norm
import numpy as np

GRACE_GRID = list(range(0, 25, 1))
TERM_DEF = 24

_Xis = []
for mi in range(M_IMP):
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=SEED + mi,
                           sample_posterior=True, initial_strategy="median")
    _Xis.append(pd.DataFrame(imp.fit_transform(d0[_cols]), columns=_cols, index=d0.index))

def _curve(expo_col, expo_within, tag):
    """For each grace value (reusing the already-imputed _Xis): build cp -> Cox -> Rubin pool."""
    out = {}
    for g in GRACE_GRID:
        est, var = [], []
        ok = True
        for mi in range(M_IMP):
            cp = build_cp_grace(d0, _Xis[mi], _cols, expo_col, expo_within, g, f"m{mi}")
            if cp["expo"].sum() == 0 or cp[cp.expo == 1]["event"].sum() == 0:
                ok = False; break
            m = CoxTimeVaryingFitter(penalizer=PENALIZER)
            m.fit(cp, id_col="id", start_col="start", stop_col="stop",
                  event_col="event", show_progress=False)
            est.append(m.summary.loc["expo", "coef"])
            var.append(m.summary.loc["expo", "se(coef)"] ** 2)
        if ok:
            hr, lo, hi, p, _, _ = pool_hr(est, var)
            out[g] = (hr, lo, hi)
    print(f"  {tag}: {len(out)}/{len(GRACE_GRID)} grace points computed")
    return out

print("=" * 60)
print(f"  continuous grace curve (0-24h, {len(GRACE_GRID)} points, MICE imputed once and reused)")
print("=" * 60)
import time; _t = time.time()
surge_curve = _curve("surge_h", DET_E, "surge")
term_curve = _curve("term_h", TERM_DEF, f"termination(≤{TERM_DEF}h)")
print(f"  elapsed: {time.time()-_t:.0f}s")
for g in GRACE_GRID:
    s = surge_curve.get(g); t = term_curve.get(g)
    _ss = f"{s[0]:.2f}[{s[1]:.2f},{s[2]:.2f}]" if s else "—"
    _tt = f"{t[0]:.2f}[{t[1]:.2f},{t[2]:.2f}]" if t else "—"
    print(f"  grace {g:>2}h | surge {_ss:>20} | term {_tt:>20}")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np

mpl.rcParams.update({
    'font.size': 8, 'axes.linewidth': 0.8, 'axes.edgecolor': '#333333',
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'figure.dpi': 300, 'savefig.dpi': 300,
})
NAVY = '#2c3e50'; BRONZE = '#8c6d3f'; REDLINE = '#b03a2e'; GREY = '#cccccc'

_surge_ids = set(MAIN_ONSET)
_cm_ids = set(cmeas["stay_id"])
_t = tmp[(tmp["stay_id"].isin(_cm_ids)) & (tmp["h"] >= 0) & (tmp["h"] <= 12)].copy()
_t["surge"] = _t["stay_id"].isin(_surge_ids).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.8), sharey=True)
grid = np.arange(0, 12.1, 1.0)
for ax, sv in zip(axes, [0, 1]):
    sub = _t[_t["surge"] == sv]
    col = NAVY if sv == 0 else BRONZE
    means = []
    for sid, g in sub.groupby("stay_id"):
        g = g.sort_values("h")
        ax.plot(g["h"], g["valuenum"], '-', color=col, lw=0.4, alpha=0.10)
        means.append(np.interp(grid, g["h"], g["valuenum"], left=np.nan, right=np.nan))
    if means:
        M = np.nanmean(np.array(means), axis=0)
        ax.plot(grid, M, '-', color=col, lw=2.0, solid_capstyle='round')
    ax.set_xlim(0, 12); ax.set_xticks([0, 4, 8, 12])
    ax.set_xlabel('Hours from CRRT initiation', fontsize=7.5)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
axes[0].set_ylabel('Transmembrane pressure (mmHg)', fontsize=7.5)
axes[0].set_title('No surge', fontsize=8)
axes[1].set_title('Surge', fontsize=8)
_ymax = np.nanpercentile(_t["valuenum"], 99)
for ax in axes: ax.set_ylim(0, _ymax)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS1.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS1.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print(f"[Fig S1] surge {int(_t.groupby('stay_id')['surge'].first().sum())} / "
      f"non-surge {int((_t.groupby('stay_id')['surge'].first()==0).sum())} trajectories")

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np
mpl.rcParams.update({'font.size':8,'axes.linewidth':0.8,'axes.edgecolor':'#333333',
    'xtick.major.width':0.8,'ytick.major.width':0.8,'xtick.color':'#333333','ytick.color':'#333333',
    'figure.dpi':300,'savefig.dpi':300})
NAVY='#2c3e50'; BRONZE='#8c6d3f'; REDLINE='#b03a2e'

def _xyz(curve):
    g = sorted(curve.keys())
    hr = np.array([curve[k][0] for k in g])
    lo = np.array([curve[k][1] for k in g])
    hi = np.array([curve[k][2] for k in g])
    return np.array(g), hr, lo, hi

fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.9), sharey=True)
for ax, curve, col, ttl in [(axes[0], surge_curve, NAVY, 'Early TMP surge'),
                            (axes[1], term_curve, BRONZE, 'Circuit termination')]:
    g, hr, lo, hi = _xyz(curve)
    ax.fill_between(g, lo, hi, color=col, alpha=0.15, linewidth=0)
    ax.plot(g, hr, '-', color=col, lw=1.6, solid_capstyle='round')
    ax.axhline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
    ax.set_yscale('log')
    ax.set_xlim(0, 24); ax.set_xticks([0, 6, 12, 18, 24])
    ax.set_xlabel('Grace period (h)', fontsize=7.5)
    ax.set_title(ttl, fontsize=8)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
axes[0].set_yticks([0.5, 1, 2, 4])
axes[0].yaxis.set_major_formatter(mticker.FixedFormatter(['0.5', '1', '2', '4']))
axes[0].yaxis.set_minor_formatter(mticker.NullFormatter())
axes[0].set_ylabel('Hazard ratio', fontsize=7.5)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS2.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS2.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print("[Fig S2] continuous grace curves saved")

fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.8), gridspec_kw={'width_ratios':[0.8,1.2]})

axA = axes[0]
vals = [surge_overlap["surge_only"], surge_overlap["overlap"], surge_overlap["filter_only"]]
ypos = [2, 1, 0]; barcols = [NAVY, '#7d6b8a', BRONZE]
axA.barh(ypos, vals, color=barcols, height=0.6, edgecolor='white')
axA.set_yticks(ypos)
axA.set_yticklabels(['Surge only', 'Overlap', 'Filter\nexhaustion only'], fontsize=7)
axA.set_xlabel('Patients (n)', fontsize=7.5)
for s in ['top', 'right']: axA.spines[s].set_visible(False)

axB = axes[1]
_keys = ['b0', 'b1', 'b2', 'b3']
b_hr = [filter_tvc[k][0] for k in _keys]
b_lo = [filter_tvc[k][1] for k in _keys]; b_hi = [filter_tvc[k][2] for k in _keys]
blab = ['Full cohort', 'Below 250 mmHg', 'Reached 250 mmHg', 'Measured TMP']
yB = np.arange(4)[::-1]
for i in range(4):
    axB.plot([b_lo[i], b_hi[i]], [yB[i], yB[i]], '-', color=NAVY, lw=1.2,
             solid_capstyle='round', alpha=0.9)
axB.scatter(b_hr, yB, c=NAVY, s=20, zorder=3, edgecolors='white', linewidths=0.5)
axB.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
axB.set_xlim(0.985, 1.020); axB.set_xticks([0.99, 1.00, 1.01])
axB.set_ylim(-0.5, 3.5); axB.set_yticks(yB); axB.set_yticklabels(blab, fontsize=7)
for s in ['top', 'right']: axB.spines[s].set_visible(False)
axB.set_xlabel('Hazard ratio per 24 h', fontsize=7.5)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS3.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS3.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print("[Fig S3] saved")

fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.8))

axA = axes[0]
axA.plot(design_N, design_power, 'o-', color=NAVY, lw=1.4, ms=4,
         markeredgecolor='white', markeredgewidth=0.4)
axA.axhline(80, ls=(0, (4, 3)), color=GREY, lw=0.8)
axA.axvline(design_N0, ls=':', color=REDLINE, lw=0.9, alpha=0.8)
axA.set_xlabel('Sample size', fontsize=7.5)
axA.set_ylabel('Power (%)', fontsize=7.5); axA.set_ylim(0, 105)
for s in ['top', 'right']: axA.spines[s].set_visible(False)

axB = axes[1]
axB.plot(design_N, design_bf10, 'o-', color=NAVY, lw=1.4, ms=4,
         markeredgecolor='white', markeredgewidth=0.4)
axB.axhline(3, ls=(0, (4, 3)), color=GREY, lw=0.8)
axB.axvline(design_N0, ls=':', color=REDLINE, lw=0.9, alpha=0.8)
axB.set_yscale('log')
axB.set_xlabel('Sample size', fontsize=7.5)
axB.set_ylabel('Bayes factor (BF10)', fontsize=7.5)
for s in ['top', 'right']: axB.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS4.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS4.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print("[Fig S4] saved")

## 5. Table reproduction and internal QC


In [ ]:
cmeas["surge_flag"] = cmeas["stay_id"].map(lambda s: 1 if s in MAIN_ONSET else 0)
c["has_base"] = c["base_median_meas"].notna().astype(int)

def _add_mortality_row(tabdf, df, group_col, ev="event28"):
    """Add a 28d-mortality row to a tab1 table (all/group0/group1 + chi-square p)."""
    al = df[ev].mean() * 100
    g0 = df[df[group_col] == 0][ev].mean() * 100
    g1 = df[df[group_col] == 1][ev].mean() * 100
    try:
        from scipy.stats import chi2_contingency
        p = chi2_contingency(pd.crosstab(df[group_col], df[ev]))[1]
    except Exception:
        p = np.nan
    row = pd.DataFrame([["28d_mortality_%", f"{al:.1f}", f"{g0:.1f}", f"{g1:.1f}", p]],
                       columns=tabdf.columns)
    return pd.concat([tabdf, row], ignore_index=True)

print("=" * 84)
print(f"  [Table 1a] by surge status (cmeas={len(cmeas)}, surge={int(cmeas['surge_flag'].sum())}, "
      f"Tier2 28d mortality={cmeas['event28'].mean()*100:.1f}%)")
print("  group=1: surge / group=0: no surge")
print("=" * 84)
t1a = tab1(cmeas, "surge_flag", CONT_M, BIN_M, cat_vars=["year_grp"])
t1a = _add_mortality_row(t1a, cmeas, "surge_flag")
print(t1a.to_string(index=False))

print("\n" + "=" * 84)
print(f"  [Table 1b] by measured-TMP status (c={len(c)}, measured={int(c['has_base'].sum())})")
print("  group=1: base_median_meas present / group=0: absent")
print("=" * 84)
t1b = tab1(c, "has_base", CONT_M, BIN_M, cat_vars=["year_grp"])
t1b = _add_mortality_row(t1b, c, "has_base")
print(t1b.to_string(index=False))
print("=" * 84)

print(f"\n  [B2 - mortality figures for the manuscript]")
print(f"    Tier 1 (c, N={len(c)}):          28d {c['event28'].mean()*100:.1f}%, "
      f"in-hospital {c['death_inhosp'].mean()*100:.1f}%")
print(f"    Tier 2 (cmeas, N={len(cmeas)}):   28d {cmeas['event28'].mean()*100:.1f}%")
print(f"      surge(+) n={int(cmeas['surge_flag'].sum())}: "
      f"28d {cmeas[cmeas.surge_flag==1]['event28'].mean()*100:.1f}%")
print(f"      surge(-) n={int((cmeas['surge_flag']==0).sum())}: "
      f"28d {cmeas[cmeas.surge_flag==0]['event28'].mean()*100:.1f}%")
print(f"    measured TMP yes {c[c.has_base==1]['event28'].mean()*100:.1f}% / "
      f"no {c[c.has_base==0]['event28'].mean()*100:.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
def _num(df, col, g):
    return df[df["surge_flag"] == g][col].dropna().median()
sa_vars = [("weight_kg", "weight"), ("anchor_age", "age"), ("sofa_total", "SOFA"),
           ("lactate", "lactate"), ("map_value", "MAP")]
x = np.arange(len(sa_vars)); w = 0.38
v0 = [_num(cmeas, c0, 0) for c0, _ in sa_vars]; v1 = [_num(cmeas, c0, 1) for c0, _ in sa_vars]
ax[0].bar(x - w/2, v0, w, label="no surge", color="#2c3e50")
ax[0].bar(x + w/2, v1, w, label="surge", color="#c0392b")
ax[0].set_xticks(x); ax[0].set_xticklabels([l for _, l in sa_vars], fontsize=9)
ax[0].set_ylabel("median"); ax[0].legend(fontsize=8)
ax[0].set_title("Table 1a: surge vs no-surge (medians)", fontsize=10, loc="left")

yc1 = c[c.has_base == 1]["year_grp"].value_counts(normalize=True).sort_index() * 100
yc0 = c[c.has_base == 0]["year_grp"].value_counts(normalize=True).reindex(yc1.index, fill_value=0) * 100
x2 = np.arange(len(yc1))
ax[1].bar(x2 - w/2, yc0.values, w, label="no measured TMP", color="#7f8c8d")
ax[1].bar(x2 + w/2, yc1.values, w, label="measured TMP", color="#16a085")
ax[1].set_xticks(x2); ax[1].set_xticklabels(yc1.index, rotation=30, ha="right", fontsize=8)
ax[1].set_ylabel("% within group"); ax[1].legend(fontsize=8)
ax[1].set_title("Table 1b: measured TMP era skew", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "table1_compare.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  saved: {FIGDIR/'table1_compare.png'}")

In [ ]:
import pandas as pd

print("="*70); print("[Table 1] by surge status - all tab1 columns"); print("="*70)
print(t1a.to_string(index=False))
print("\n" + "="*70); print("[Table S1] by measured status - all tab1 columns"); print("="*70)
print(t1b.to_string(index=False))

print("\n" + "="*70); print("[supplement] continuous variables, median (Q1-Q3) by group - Table 1 (cmeas, surge_flag)"); print("="*70)
for col in CONT_M:
    if col not in cmeas.columns: continue
    for g, gname in [(0,"no surge"), (1,"surge")]:
        v = cmeas[cmeas["surge_flag"]==g][col].dropna()
        if len(v):
            print(f"  {col:<16} {gname:<9} {v.median():.1f} ({v.quantile(.25):.1f}-{v.quantile(.75):.1f})  n={len(v)}")
    v = cmeas[col].dropna()
    print(f"  {col:<16} {'overall':<9} {v.median():.1f} ({v.quantile(.25):.1f}-{v.quantile(.75):.1f})  n={len(v)}")
    print()

print("="*70); print("[supplement] binary variables, n (%) by group - Table 1"); print("="*70)
for col in BIN_M:
    if col not in cmeas.columns: continue
    row = f"  {col:<20}"
    for g, gname in [(None,"overall"), (0,"no surge"), (1,"surge")]:
        d = cmeas if g is None else cmeas[cmeas["surge_flag"]==g]
        n1 = int(d[col].sum()); N = len(d)
        row += f" | {gname} {n1}/{N} ({n1/N*100:.0f}%)"
    print(row)

In [ ]:
print("="*90); print("[Table S3] 50 definitions"); print("="*90)
_cols_show = ["baseline","rise","window","n_surge","HR","lo","hi","p","q"]
_cols_show = [c for c in _cols_show if c in resdf.columns]
_show = resdf[_cols_show].copy()
_show = _show.sort_values(["baseline","rise","window"],
                          key=lambda s: s.map({"median":0,"first":1}) if s.name=="baseline" else s)
for col in ["HR","lo","hi"]:
    if col in _show: _show[col] = _show[col].round(3)
for col in ["p","q"]:
    if col in _show: _show[col] = _show[col].round(3)
print(_show.to_string(index=False))

In [ ]:
from statsmodels.stats.multitest import multipletests
_r = resdf.copy()
for bm in ["median","first"]:
    mask = _r["baseline"]==bm
    _r.loc[mask,"q"] = multipletests(_r.loc[mask,"p"], method="fdr_bh")[1]
print(_r[["baseline","rise","window","n_surge","HR","lo","hi","p","q"]].round(3).to_string(index=False))

In [ ]:
import pandas as pd
print("="*78); print("[Table S2] MIMIC-IV 12 settings (r_l1m)"); print("="*78)
_m = r_l1m.copy()
_m["Endpoint"] = _m["endpoint"].map({"28d":"28-day","in-hosp":"In-hospital"}).fillna(_m["endpoint"])
_m["Source"] = _m["source"].map(lambda s: "Procedure" if str(s).startswith("proc") else "Union")
_m = _m[["Endpoint","Source","gap","N","ev","HR","lo","hi","p"]]
_m.columns = ["Endpoint","Source","Gap(h)","N","Events","HR","lo","hi","P"]
for col in ["HR","lo","hi"]: _m[col]=_m[col].round(3)
_m["P"]=_m["P"].round(3)
print(_m.to_string(index=False))

print("\n"+"="*78); print("[Table S2] eICU 4 settings (r_l1e)"); print("="*78)
_e = r_l1e.copy()
_e["Setting"] = _e.apply(lambda r: f"≥{int(r['min_rec'])} records, "
                         f"{'last' if str(r['term']).startswith('a') else '+gap'}", axis=1)
_e = _e[["Setting","N","ev","HR","lo","hi","p"]]
_e.columns = ["Setting","N","Events","HR","lo","hi","P"]
for col in ["HR","lo","hi"]: _e[col]=_e[col].round(3)
_e["P"]=_e["P"].round(3)
print(_e.to_string(index=False))

print("\n"+"="*78); print("[Table S1] by measured-TMP status (c, has_base) - with IQR"); print("="*78)
from scipy.stats import mannwhitneyu, chi2_contingency
g1 = c[c["has_base"]==1]; g0 = c[c["has_base"]==0]
print(f"  With measured (n={len(g1)}) vs Without (n={len(g0)})\n")

print("  -- continuous: median (Q1-Q3) --")
for col in CONT_M:
    if col not in c.columns: continue
    v1 = g1[col].dropna(); v0 = g0[col].dropna()
    if len(v1)<2 or len(v0)<2: continue
    try: _, p = mannwhitneyu(v1, v0, alternative="two-sided")
    except: p = float("nan")
    print(f"  {col:<16} With {v1.median():>6.1f} ({v1.quantile(.25):.1f}-{v1.quantile(.75):.1f})"
          f"  | Without {v0.median():>6.1f} ({v0.quantile(.25):.1f}-{v0.quantile(.75):.1f})"
          f"  | p={p:.3g}")

print("\n  -- binary: n (%) --")
for col in BIN_M:
    if col not in c.columns: continue
    n1=int(g1[col].sum()); n0=int(g0[col].sum())
    try:
        ct = pd.crosstab(c["has_base"], c[col])
        _, p, _, _ = chi2_contingency(ct)
    except: p=float("nan")
    print(f"  {col:<18} With {n1}/{len(g1)} ({n1/len(g1)*100:.0f}%)"
          f"  | Without {n0}/{len(g0)} ({n0/len(g0)*100:.0f}%)  | p={p:.3g}")

print("\n  -- Admission 2014+ --")
for lab, g in [("With", g1), ("Without", g0)]:
    if "year_grp" in g.columns:
        late = g["year_grp"].astype(str).str.contains("2014|2015|2016|2017|2018|2019", regex=True).sum()
        print(f"  {lab}: {int(late)}/{len(g)} ({late/len(g)*100:.0f}%)")

In [ ]:
print("="*60)
print("[7-8] checking heparin 367/367, prophylaxis 184/185")
print("="*60)

t2 = cmeas
mb = c[c["has_base"]==1]

print("\n--- systemic heparin ---")
print(f"Tier2(862) systemic: {int(t2['v3_systemic_hep'].sum())} / {len(t2)} = {t2['v3_systemic_hep'].mean()*100:.1f}%")
print(f"measured(873) systemic: {int(mb['v3_systemic_hep'].sum())} / {len(mb)} = {mb['v3_systemic_hep'].mean()*100:.1f}%")

print("\n--- prophylaxis heparin ---")
print(f"Tier2(862) prophylaxis: {int(t2['v3_prophylaxis'].sum())} / {len(t2)} = {t2['v3_prophylaxis'].mean()*100:.1f}%")
print(f"measured(873) prophylaxis: {int(mb['v3_prophylaxis'].sum())} / {len(mb)} = {mb['v3_prophylaxis'].mean()*100:.1f}%")

_lost = mb[~mb["stay_id"].isin(set(t2["stay_id"]))]
print(f"\nOf {len(_lost)} patients excluded going from 873 to 862:")
print(f"  on systemic heparin: {int(_lost['v3_systemic_hep'].sum())}")
print(f"  on prophylactic heparin: {int(_lost['v3_prophylaxis'].sum())}")

In [ ]:
print("="*60)
print("[9] operating-time 28d P: abstract .16 vs main text .164")
print("="*60)
_main28 = r_l1m[(r_l1m["source"].str.startswith("proc")) & 
                (r_l1m["gap"]==6) & (r_l1m["endpoint"]=="28d")].iloc[0]
print(f"HR={_main28['HR']:.4f}, P={_main28['p']:.4f}")
print(f"Abstract: P=.16 (rounded), main text: P=.164")
print("-> same value, different rounding, as expected" if abs(_main28['p']-0.164)<0.001 else "-> discrepancy needs checking")

In [ ]:
print("="*60)
print("[Table S5] sensitivity-analysis values")
print("="*60)

def _fmt(name, n, hr, lo, hi, p, ev=None):
    evs = f"{ev}" if ev is not None else "—"
    print(f"  {name:<38} N={n:<5} ev={evs:<5} HR={hr:.2f} [{lo:.2f}-{hi:.2f}] P={p:.3f}")

print("\n-- Primary (fully adjusted, M5) --")
try:
    print(f"  r_main: HR={r_main[0]:.3f} [{r_main[1]:.3f}-{r_main[2]:.3f}] p={r_main[3]:.3f}")
except Exception as e:
    print("  r_main not found:", e)

print("\n-- searching for the cell-8 sensitivity variables --")
import re
_cand = [v for v in dir() if re.search(r'sens|sepsis|complete|cc|d7|seven|whole|baseTMP|btmp|evalue|eval', v, re.I)]
print("  candidates:", _cand)

for vn in ['sens_tab','r_sens','sensdf','S5','tab_s5','r_s5']:
    if vn in dir():
        print(f"\n  found {vn}:")
        print(eval(vn).to_string(index=False) if hasattr(eval(vn),'to_string') else eval(vn))

In [ ]:
print("="*60)
print("[Table S5] extracting exact sensitivity values")
print("="*60)

def _show(name, obj):
    try:
        hr,lo,hi,p = obj[0],obj[1],obj[2],obj[3]
        print(f"  {name:<32} HR={hr:.3f} [{lo:.3f}-{hi:.3f}] p={p:.3f}")
    except Exception as e:
        print(f"  {name:<32} = {obj}  ({type(obj).__name__})")

print("\n-- each sensitivity result --")
for vn in ['r_main','r_sepsis_s','r_sepsis_b','r_complete']:
    if vn in dir(): _show(vn, eval(vn))

print("\n-- searching for additional variables (7day/baseTMP/whole) --")
import re
_more = [v for v in dir() if re.search(r'd7|day7|seven|btmp|basetmp|tmp_adj|whole|r_w', v, re.I)]
print("  candidates:", _more)
for vn in _more:
    try: _show(vn, eval(vn))
    except: pass

print("\n-- E-value --")
if 'evalue' in dir():
    print(f"  evalue = {evalue}  ({type(evalue).__name__})")

print("\n-- subgroup N / events --")
if '_need_sepsis' in dir() or 'r_sepsis_s' in dir():
    try:
        print(f"  sepsis-strict: r_sepsis_s = {r_sepsis_s}")
    except: pass
if 'cc' in dir():
    try:
        print(f"  complete-case n={len(cc)}, 28d deaths={int(cc['event28'].sum()) if 'event28' in cc.columns else '?'}")
    except Exception as e:
        print("  cc:", e)
if '_cc_mask' in dir():
    try:
        print(f"  _cc_mask sum (n)={int(_cc_mask.sum())}")
    except: pass

In [ ]:
print("sepsis-strict(590) deaths:", end=" ")
try:
    _ss = cmeas[(cmeas.get('_sepsis_strict', cmeas.get('sepsis_strict'))==1)] if '_sepsis_strict' in cmeas.columns or 'sepsis_strict' in cmeas.columns else None
    print(int(_ss['event28'].sum()) if _ss is not None else "variable name needs checking")
except Exception as e:
    print(e)

print("complete-case(576) deaths:", int(cmeas[_cc_mask]['event28'].sum()) if '_cc_mask' in dir() else "?")

print("primary(862) 7-day deaths:", int(cmeas['event7'].sum()) if 'event7' in cmeas.columns else "check the event7 variable name")

## 6. Revision-response analyses


In [ ]:
import numpy as np, pandas as pd
from lifelines import CoxTimeVaryingFitter

B_BOOT   = 500
B_BOOT_T1 = 200
BOOT_SEED = 20260629

_CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C
_BIN_M5  = DEMO_B + SEV_B + TX_B
_cont_m5 = [c_ for c_ in _CONT_M5 if c_ in cmeas.columns or c_ in XC_MEAS[0].columns]
_bin_m5  = [c_ for c_ in _BIN_M5 if c_ in cmeas.columns]

def _tvc_coef_once(d_boot, Xc_boot, onset_map, cont, binc, exposure,
                   builder, segs=None, end_col="end28_h", ev_col="event28",
                   penalizer=None):
    """One bootstrap replicate: return the exposure coefficient, or None if the fit fails."""
    pen = PENALIZER if penalizer is None else penalizer
    if builder == "surge":
        cp = build(d_boot, Xc_boot, onset_map, cont, binc, end_col, ev_col)
        expo = "surge"
    else:
        cp = build_grid_mimic(d_boot, Xc_boot, segs, cont, binc, end_col, ev_col)
        expo = "cum_dur_h"
    if len(cp) == 0 or cp["event"].sum() == 0:
        return None
    if builder == "surge" and (cp["surge"].sum() == 0
                               or cp[cp.surge == 1]["event"].sum() == 0):
        return None
    fit = [expo] + [cc for cc in (cont + binc) if cp[cc].nunique() > 1]
    try:
        m = CoxTimeVaryingFitter(penalizer=pen)
        m.fit(cp[["id", "start", "stop", "event"] + fit], id_col="id",
              start_col="start", stop_col="stop", event_col="event",
              show_progress=False)
        return float(m.summary.loc[expo, "coef"])
    except Exception:
        return None

def cluster_bootstrap_se(d, Xc_full, onset_map, cont, binc, exposure, builder,
                         segs=None, end_col="end28_h", ev_col="event28",
                         scale=1.0, B=B_BOOT, seed=BOOT_SEED):
    """Patient-level bootstrap (resampling with replacement) for the exposure coefficient's
       SE (log scale) and percentile CI. Resampled patients get new integer ids to avoid
       duplicate (start, stop] rows."""
    rng = np.random.default_rng(seed)
    n = len(d)
    d = d.reset_index(drop=True)
    Xc_full = Xc_full.reset_index(drop=True)
    coefs = []
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        d_b = d.iloc[idx].copy().reset_index(drop=True)
        new_ids = np.arange(len(d_b))
        old_ids = d_b["stay_id"].values.copy()
        d_b["stay_id"] = new_ids
        Xc_b = Xc_full.iloc[idx].reset_index(drop=True)
        if builder == "mimic":
            segs_b = {int(new_ids[k]): segs.get(int(old_ids[k]), [])
                      for k in range(len(d_b))}
            cf = _tvc_coef_once(d_b, Xc_b, None, cont, binc, exposure,
                                "mimic", segs=segs_b, end_col=end_col, ev_col=ev_col)
        else:
            onset_b = {int(new_ids[k]): onset_map.get(int(old_ids[k]), np.nan)
                       for k in range(len(d_b))}
            cf = _tvc_coef_once(d_b, Xc_b, onset_b, cont, binc, exposure,
                                "surge", end_col=end_col, ev_col=ev_col)
        if cf is not None:
            coefs.append(cf)
    coefs = np.array(coefs)
    se = coefs.std(ddof=1)
    lo = np.exp(np.percentile(coefs, 2.5) * scale)
    hi = np.exp(np.percentile(coefs, 97.5) * scale)
    return se, lo, hi, len(coefs)

print("=" * 72)
print("  [2.1] cluster (patient)-robust SE via bootstrap (B=%d)" % B_BOOT)
print("  tie method: Breslow (fixed by CoxTimeVaryingFitter; stated in Methods)")
print("=" * 72)

_cp0 = build(cmeas, XC_MEAS[0][_cont_m5], MAIN_ONSET, _cont_m5, _bin_m5)
_fit = ["surge"] + [cc for cc in (_cont_m5 + _bin_m5) if _cp0[cc].nunique() > 1]
_m0 = CoxTimeVaryingFitter(penalizer=PENALIZER)
_m0.fit(_cp0[["id", "start", "stop", "event"] + _fit], id_col="id",
        start_col="start", stop_col="stop", event_col="event", show_progress=False)
_coef0 = float(_m0.summary.loc["surge", "coef"])
_se_model = float(_m0.summary.loc["surge", "se(coef)"])

_se_boot, _lo_b, _hi_b, _nb = cluster_bootstrap_se(
    cmeas, XC_MEAS[0], MAIN_ONSET, _cont_m5, _bin_m5, "surge", "surge")

print("\n[Tier 2 surge M5]")
print(f"  model SE (single MICE set) = {_se_model:.4f}")
print(f"  bootstrap cluster SE       = {_se_boot:.4f}  (ratio={_se_boot/_se_model:.3f}, n_ok={_nb})")
print(f"  point HR = {np.exp(_coef0):.3f}")
print(f"  bootstrap 95% CI (percentile) = [{_lo_b:.3f}, {_hi_b:.3f}]")
print(f"  Note: primary result r_main (Rubin-pooled) HR={r_main[0]:.3f} "
      f"[{r_main[1]:.3f},{r_main[2]:.3f}]")

_cont_t1 = [c_ for c_ in CONT_M if c_ in c.columns or c_ in XC_C[0].columns]
_bin_t1  = [c_ for c_ in BIN_M if c_ in c.columns]
_cpg = build_grid_mimic(c, XC_C[0][_cont_t1], SEG_MAIN, _cont_t1, _bin_t1,
                        "end28_h", "event28")
_fitg = ["cum_dur_h"] + [cc for cc in (_cont_t1 + _bin_t1) if _cpg[cc].nunique() > 1]
_mg = CoxTimeVaryingFitter(penalizer=PENALIZER)
_mg.fit(_cpg[["id", "start", "stop", "event"] + _fitg], id_col="id",
        start_col="start", stop_col="stop", event_col="event", show_progress=False)
_coefg = float(_mg.summary.loc["cum_dur_h", "coef"])
_se_model_g = float(_mg.summary.loc["cum_dur_h", "se(coef)"])

_se_boot_g, _lo_g, _hi_g, _nbg = cluster_bootstrap_se(
    c, XC_C[0], None, _cont_t1, _bin_t1, "cum_dur_h", "mimic",
    segs=SEG_MAIN, end_col="end28_h", ev_col="event28", scale=24, B=B_BOOT_T1)

print("\n[Tier 1 cumulative operating time, per 24h]")
print(f"  model SE (per-hour coef)   = {_se_model_g:.5f}")
print(f"  bootstrap cluster SE       = {_se_boot_g:.5f}  (ratio={_se_boot_g/_se_model_g:.3f}, n_ok={_nbg})")
print(f"  point HR per 24h = {np.exp(_coefg*24):.4f}")
print(f"  bootstrap 95% CI (per 24h) = [{_lo_g:.4f}, {_hi_g:.4f}]")
print("=" * 72)
print("interpretation:")
print("  ratio~1 -> no SE underestimation from within-patient correlation -> supports the primary result.")
print("  if the ratio grows past ~1.15 and the CI upper bound crosses 1 -> report the robust (bootstrap) CI")
print("            as the primary result and emphasize the exploratory nature of the surge signal.")
print("=" * 72)

robust_se_tab = {
    "tier2_surge": {"HR": np.exp(_coef0), "se_model": _se_model,
                    "se_boot": _se_boot, "ci_boot": (_lo_b, _hi_b), "n_boot": _nb},
    "tier1_per24h": {"HR": np.exp(_coefg*24), "se_model": _se_model_g,
                     "se_boot": _se_boot_g, "ci_boot": (_lo_g, _hi_g), "n_boot": _nbg},
}

In [ ]:
import numpy as np, pandas as pd

M_HIGH = 20

_CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C
_BIN_M5  = DEMO_B + SEV_B + TX_B
_cont_m5 = [x for x in _CONT_M5 if x in cmeas.columns or x in XC_MEAS[0].columns]
_bin_m5  = [x for x in _BIN_M5 if x in cmeas.columns]

print("=" * 72)
print("  [2.8] multiple imputation m=5 vs m=%d (Monte Carlo stability)" % M_HIGH)
print("=" * 72)

print("\n[Tier 2 continuous-covariate missingness (cmeas=%d)]" % len(cmeas))
miss = {}
for v in _cont_m5:
    if v in cmeas.columns:
        r = float(cmeas[v].isna().mean()) * 100
        miss[v] = r
        print(f"    {v:<14} {r:5.1f}%")
_maxmiss = max(miss.values()) if miss else 0.0
print(f"  max missingness = {_maxmiss:.1f}%  "
      f"-> {'low (m=5 is usually adequate; m=20 recommended to confirm)' if _maxmiss < 20 else 'high (m=20-50 needed)'}")

XC_MEAS_20 = mice_impute(cmeas, _cont_m5, "event28", "end28_h", m=M_HIGH)

r_m5_5  = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, _cont_m5, _bin_m5)
r_m5_20 = fit_surge_cox(cmeas, XC_MEAS_20, MAIN_ONSET, _cont_m5, _bin_m5)

def _mc_between_var(d, XC_list, onset, cont, binc):
    """Between-imputation variance B (log HR) only, as a Monte Carlo error indicator."""
    ests = []
    for mi in range(len(XC_list)):
        Xc = XC_list[mi][cont]
        cp = build(d, Xc, onset, cont, binc)
        fit = ["surge"] + [cc for cc in (cont + binc) if cp[cc].nunique() > 1]
        from lifelines import CoxTimeVaryingFitter
        m = CoxTimeVaryingFitter(penalizer=PENALIZER)
        m.fit(cp[["id", "start", "stop", "event"] + fit], id_col="id",
              start_col="start", stop_col="stop", event_col="event",
              show_progress=False)
        ests.append(m.summary.loc["surge", "coef"])
    return np.var(ests, ddof=1), np.mean(ests)

B5, Q5   = _mc_between_var(cmeas, XC_MEAS,    MAIN_ONSET, _cont_m5, _bin_m5)
B20, Q20 = _mc_between_var(cmeas, XC_MEAS_20, MAIN_ONSET, _cont_m5, _bin_m5)

print("\n[primary surge M5]")
print(f"  m= 5 : HR={r_m5_5[0]:.3f} [{r_m5_5[1]:.3f},{r_m5_5[2]:.3f}] p={r_m5_5[3]:.3f}")
print(f"  m=20 : HR={r_m5_20[0]:.3f} [{r_m5_20[1]:.3f},{r_m5_20[2]:.3f}] p={r_m5_20[3]:.3f}")
print(f"  between-set variance B (log HR): m=5={B5:.5f}  m=20={B20:.5f}")
print(f"  MC standard error ~ sqrt(B/m): m=5={np.sqrt(B5/5):.4f}  m=20={np.sqrt(B20/20):.4f}")
print("=" * 72)
print("interpretation: if HR/CI are essentially unchanged from m=5 to m=20, the m=5 result is MC-stable.")
print("      reporting plan: promote m=20 to the primary analysis, footnote that m=5 and m=20 agree.")
print("=" * 72)

imp_m_tab = {"m5": r_m5_5, "m20": r_m5_20, "B5": B5, "B20": B20, "missrate": miss}

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import chi2
from lifelines import CoxPHFitter

def _baseline_median(g):
    b = g[g["h"] <= 3]["valuenum"]
    return b.median() if len(b) else np.nan

def _max_delta_12h(g):
    base = _baseline_median(g)
    w = g[(g["h"] >= 0) & (g["h"] <= LM_H)]
    if len(w) == 0 or pd.isna(base): return np.nan
    return float(w["valuenum"].max() - base)

_dmax = tmp.groupby("stay_id").apply(_max_delta_12h)
_dmax = _dmax.dropna().rename("max_dtmp").reset_index()

if "lm" in dir() and isinstance(lm, pd.DataFrame) and "fu_h" in lm.columns:
    base_lm = lm.copy()
else:
    _cl = c.merge(_dmax, on="stay_id", how="left")
    _cl["died_28d"] = _cl["event28"]
    b = _cl[_cl["max_dtmp"].notna()].copy()
    b = b[(b["death_h"].isna()) | (b["death_h"] > LM_H)]
    b = b[(b["disch_h"].isna()) | (b["disch_h"] > LM_H)]
    _end = np.minimum(b["death_h"].fillna(np.inf), FU_H)
    _cz = (b["died_28d"] == 0) & b["disch_h"].notna() & (b["disch_h"] < _end)
    _end = _end.copy(); _end[_cz] = b["disch_h"][_cz]
    b["fu_h"] = _end - LM_H
    b["event"] = ((b["died_28d"] == 1) & (b["death_h"] <= FU_H)).astype(int)
    base_lm = b[b["fu_h"] > 0].reset_index(drop=True)

if "max_dtmp" not in base_lm.columns:
    base_lm = base_lm.merge(_dmax, on="stay_id", how="left")
d_rcs = base_lm[base_lm["max_dtmp"].notna()].reset_index(drop=True)
print("=" * 72)
print("  [2.4] max ΔTMP restricted cubic spline (12h landmark)")
print("=" * 72)
print(f"  eligible = 12h survivors with a computable max delta-TMP: N={len(d_rcs)}, 28d deaths={int(d_rcs['event'].sum())}")
print(f"  max delta-TMP distribution: median={d_rcs['max_dtmp'].median():.1f}, "
      f"IQR[{d_rcs['max_dtmp'].quantile(.25):.1f}, {d_rcs['max_dtmp'].quantile(.75):.1f}], "
      f"range [{d_rcs['max_dtmp'].min():.1f}, {d_rcs['max_dtmp'].max():.1f}]")

CONT_R = [x for x in CONT_M if x in d_rcs.columns and not d_rcs[x].isna().all()]
BIN_R  = [x for x in BIN_M if x in d_rcs.columns]
for x in BIN_R:
    d_rcs[x] = pd.to_numeric(d_rcs[x], errors="coerce").fillna(0).astype(int)
XC_R = mice_impute(d_rcs, CONT_R, "event", "fu_h", m=M_IMP)

KNOTS = np.quantile(d_rcs["max_dtmp"], [0.05, 0.35, 0.65, 0.95])
_k1, _k2, _k3, _k4 = KNOTS
GRID = np.linspace(max(0.0, d_rcs["max_dtmp"].min()),
                   min(250, d_rcs["max_dtmp"].max()), 61)

def _rcs_design(x, ref_di=None):
    """Harrell RMS 4-knot restricted cubic spline basis, fixed knots,
       normalized by (k4-k1)^2. ref_di is unused (kept for API compatibility)."""
    x = np.asarray(x, float)
    def tp(u, k): return np.maximum(u - k, 0.0) ** 3
    denom = (_k4 - _k1) ** 2
    s1 = (tp(x, _k1) - tp(x, _k3) * (_k4 - _k1) / (_k4 - _k3)
          + tp(x, _k4) * (_k3 - _k1) / (_k4 - _k3)) / denom
    s2 = (tp(x, _k2) - tp(x, _k3) * (_k4 - _k2) / (_k4 - _k3)
          + tp(x, _k4) * (_k3 - _k2) / (_k4 - _k3)) / denom
    return np.column_stack([x, s1, s2]), None

grid_lhr = []
lr_pvals = []
for mi in range(len(XC_R)):
    dd = d_rcs[["fu_h", "event", "max_dtmp"]].copy()
    for cc in CONT_R: dd[cc] = XC_R[mi][cc].values
    for cc in BIN_R:  dd[cc] = d_rcs[cc].values
    Bmat, di = _rcs_design(dd["max_dtmp"].values)
    bcols = [f"rcs{j}" for j in range(Bmat.shape[1])]
    for j, cn in enumerate(bcols): dd[cn] = Bmat[:, j]
    covs = bcols + [c_ for c_ in (CONT_R + BIN_R) if dd[c_].nunique() > 1]
    dd_fit = dd[["fu_h", "event"] + covs].copy()
    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(dd_fit, duration_col="fu_h", event_col="event")
    dd_lin = dd[["fu_h", "event", "max_dtmp"]
                + [c_ for c_ in (CONT_R + BIN_R) if dd[c_].nunique() > 1]].copy()
    cph_lin = CoxPHFitter(penalizer=0.01)
    cph_lin.fit(dd_lin, duration_col="fu_h", event_col="event")
    ll_full = cph.log_likelihood_; ll_lin = cph_lin.log_likelihood_
    dfnl = len(bcols) - 1
    lr = 2 * (ll_full - ll_lin)
    lr_pvals.append(1 - chi2.cdf(max(lr, 0), dfnl))
    Bg, _ = _rcs_design(GRID)
    B0, _ = _rcs_design(np.array([0.0]))
    beta = np.array([cph.params_[cn] for cn in bcols])
    lhr = (Bg - B0) @ beta
    grid_lhr.append(lhr)

grid_lhr = np.array(grid_lhr)
lhr_mean = grid_lhr.mean(axis=0)
lhr_lo = np.percentile(grid_lhr, 2.5, axis=0)
lhr_hi = np.percentile(grid_lhr, 97.5, axis=0)
hr_curve = np.exp(lhr_mean)

print(f"\n  RCS 4 knots (Harrell, 5/35/65/95 pct of max ΔTMP) = "
      f"[{_k1:.1f}, {_k2:.1f}, {_k3:.1f}, {_k4:.1f}]")
print(f"  nonlinearity test (spline vs linear), mean p across MICE sets = {np.mean(lr_pvals):.3f}"
      f"  [{'nonlinearity significant' if np.mean(lr_pvals)<0.05 else 'weak evidence of nonlinearity (near-monotonic/linear)'}]")
_hi = d_rcs["max_dtmp"].max()
print(f"  curve HR (ref = delta-TMP 0, observed max {_hi:.0f}):")
for xq in [0, 50, 100, 150, 200, 250]:
    if xq > GRID.max():
        continue
    j = int(np.argmin(np.abs(GRID - xq)))
    print(f"    ΔTMP={GRID[j]:6.1f} → HR={hr_curve[j]:.3f} "
          f"[{np.exp(lhr_lo[j]):.3f}, {np.exp(lhr_hi[j]):.3f}]")
print("=" * 72)
print("interpretation:")
print("  - nonlinear p<0.05 and the curve steepens with rise -> nonlinear dose-response.")
print("  - curve rises monotonically above baseline (large nonlinear p) -> a near-linear dose-response;")
print("    the surge dichotomization can be described as a discrete approximation of this continuous relation.")
print("  - a flat/null curve would suggest the surge signal is an artifact of measurement or selection.")
print("=" * 72)

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.2, 3.6))
    ax.plot(GRID, hr_curve, color="#1f3a5f", lw=2)
    ax.fill_between(GRID, np.exp(lhr_lo), np.exp(lhr_hi), color="#1f3a5f", alpha=0.15)
    ax.axhline(1.0, color="grey", lw=0.8, ls="--")
    ax.set_xlim(0, 250)
    ax.set_xlabel("Maximum ΔTMP within 12 h (mmHg)")
    ax.set_ylabel("Hazard ratio (ref = no rise)")
    ax.set_title("Dose-response: early ΔTMP and 28-day mortality (RCS)")
    fig.tight_layout()
    fig.savefig(FIGDIR / "rcs_dtmp_doseresponse.png", dpi=200)
    fig.savefig(FIGDIR / "rcs_dtmp_doseresponse.pdf")
    plt.close(fig)
    print("  [figure saved] figs/rcs_dtmp_doseresponse.png/.pdf")
except Exception as e:
    print("  (figure skipped:", e, ")")

rcs_tab = {"grid": GRID, "hr": hr_curve, "lo": np.exp(lhr_lo), "hi": np.exp(lhr_hi),
           "nonlin_p": float(np.mean(lr_pvals)), "knots": KNOTS.tolist(),
           "N": len(d_rcs), "events": int(d_rcs["event"].sum())}

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

print("=" * 72)
print("  [2.6] early 12h TMP measurement intensity: surge(+) vs surge(-) (descriptive)")
print("=" * 72)

_sids = set(cmeas["stay_id"].astype(int))
t12 = tmp[(tmp["stay_id"].astype(int).isin(_sids)) &
          (tmp["h"] >= 0) & (tmp["h"] <= LM_H)].copy()

def _meas_feats(g):
    h = np.sort(g["h"].values)
    n = len(h)
    first = h[0] if n else np.nan
    gap = np.median(np.diff(h)) if n >= 2 else np.nan
    return pd.Series({"n_meas": n, "first_h": first, "med_gap_h": gap})

feat = t12.groupby("stay_id").apply(_meas_feats).reset_index()
feat["surge"] = feat["stay_id"].map(
    lambda s: 1 if pd.notna(MAIN_ONSET.get(int(s), np.nan)) else 0)

_all = pd.DataFrame({"stay_id": list(_sids)})
feat = _all.merge(feat, on="stay_id", how="left")
feat["n_meas"] = feat["n_meas"].fillna(0)
feat["surge"] = feat["stay_id"].map(
    lambda s: 1 if pd.notna(MAIN_ONSET.get(int(s), np.nan)) else 0)

def _summ(col):
    a = feat[feat.surge == 1][col].dropna()
    b = feat[feat.surge == 0][col].dropna()
    if len(a) >= 3 and len(b) >= 3:
        try:
            p = mannwhitneyu(a, b, alternative="two-sided").pvalue
        except Exception:
            p = np.nan
    else:
        p = np.nan
    return (a.median(), a.quantile(.25), a.quantile(.75),
            b.median(), b.quantile(.25), b.quantile(.75), p)

print(f"\n  surge(+) n={int((feat.surge==1).sum())}  |  surge(-) n={int((feat.surge==0).sum())}")
print(f"\n  {'metric':<22}{'surge(+) med[IQR]':<26}{'surge(-) med[IQR]':<26}{'p(MWU)'}")
for col, lab in [("n_meas", "12h measurement count"),
                 ("first_h", "first measurement time (h)"),
                 ("med_gap_h", "median measurement gap (h)")]:
    a_m, a_lo, a_hi, b_m, b_lo, b_hi, p = _summ(col)
    print(f"  {lab:<22}{a_m:5.1f} [{a_lo:.1f},{a_hi:.1f}]        "
          f"{b_m:5.1f} [{b_lo:.1f},{b_hi:.1f}]        "
          f"{p:.3f}" if not np.isnan(p) else
          f"  {lab:<22}(insufficient sample)")
print("=" * 72)
print("interpretation:")
print("  - if measurement count differs significantly and surge(+) is measured much more often, note the possibility of")
print("    monitoring-intensity confounding as a limitation, supporting the exploratory framing of surge.")
print("  - a small difference argues against measurement intensity driving the surge finding.")
print("  - inverse-intensity weighting would be unstable with 46 events, so this is reported descriptively only.")
print("=" * 72)

meas_intensity_tab = feat

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from lifelines import CoxTimeVaryingFitter

print("=" * 72)
print("  [2.5] IP-of-observation weighting (measurement selection bias) - confirmatory sensitivity")
print("=" * 72)

_cm_ids = set(cmeas["stay_id"].astype(int))
sel = c.copy()
sel["measured"] = sel["stay_id"].astype(int).isin(_cm_ids).astype(int)

def _total_on(sid):
    segs = SEG_MAIN.get(int(sid), [])
    return sum(e - s for s, e in segs) if segs else 0.0
sel["cum_on_total"] = sel["stay_id"].map(_total_on)

_pred_cont = [x for x in ["anchor_age", "weight_kg", "sofa_total", "blood_flow",
                          "cum_on_total"] if x in sel.columns]
_pred_bin  = [x for x in ["male", "vaso_use", "mech_vent"] if x in sel.columns]
if "year_grp" in sel.columns:
    yd = pd.get_dummies(sel["year_grp"].astype(str), prefix="yr", drop_first=True)
else:
    yd = pd.DataFrame(index=sel.index)

Xdf = sel[_pred_cont].copy()
for cc in _pred_cont: Xdf[cc] = Xdf[cc].fillna(Xdf[cc].median())
for cc in _pred_bin:
    Xdf[cc] = pd.to_numeric(sel[cc], errors="coerce").fillna(0).astype(int)
Xdf = pd.concat([Xdf, yd.astype(int)], axis=1)
y = sel["measured"].values

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(Xdf.values, y)
ps = lr.predict_proba(Xdf.values)[:, 1]
ps = np.clip(ps, 1e-3, 1 - 1e-3)
p_marg = y.mean()

sel["ps"] = ps
sel["sw"] = np.where(sel["measured"] == 1, p_marg / sel["ps"], np.nan)
_sw_meas = sel.loc[sel.measured == 1, "sw"]
lo_t, hi_t = np.percentile(_sw_meas.dropna(), [1, 99])
sel["sw_tr"] = sel["sw"].clip(lo_t, hi_t)

w_map = dict(zip(sel["stay_id"].astype(int), sel["sw_tr"]))
w_vec = cmeas["stay_id"].astype(int).map(w_map).fillna(1.0).values

w = w_vec
ESS = (w.sum() ** 2) / (np.square(w).sum())
from sklearn.metrics import roc_auc_score
_sel_auc = roc_auc_score(y, ps)
print("\n[A. selection-model diagnostics]")
print(f"  measured proportion = {p_marg:.3f} (cmeas {int(y.sum())}/{len(y)})")
print(f"  predictors: {_pred_cont + _pred_bin + list(yd.columns)}")
print(f"  selection-model AUC (training sample, optimistic) = {_sel_auc:.3f}"
      f"  [higher = the measured group is more distinctly selected -> selection bias is a real concern]")
print("\n[B. stabilized-weight distribution (cmeas, truncated at 1st/99th pct)]")
print(f"  min={w.min():.3f}  max={w.max():.3f}  mean={w.mean():.3f}  "
      f"CV={w.std()/w.mean():.3f}")
print(f"  effective sample size ESS = {ESS:.0f} / {len(w)}  (reduction {100*(1-ESS/len(w)):.1f}%)")
print(f"  also worth checking the weight distribution among surge(+) (sensitive with only 46 patients).")

_CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C
_BIN_M5  = DEMO_B + SEV_B + TX_B
_cont_m5 = [x for x in _CONT_M5 if x in cmeas.columns or x in XC_MEAS[0].columns]
_bin_m5  = [x for x in _BIN_M5 if x in cmeas.columns]

def _weighted_surge_hr(d, Xc, onset, cont, binc, wmap, penalizer=PENALIZER):
    cp = build(d, Xc, onset, cont, binc)
    if len(cp) == 0 or cp["surge"].sum() == 0: return None
    cp["w"] = cp["id"].map(wmap).fillna(1.0)
    fit = ["surge"] + [cc for cc in (cont + binc) if cp[cc].nunique() > 1]
    m = CoxTimeVaryingFitter(penalizer=penalizer)
    m.fit(cp[["id", "start", "stop", "event", "w"] + fit], id_col="id",
          start_col="start", stop_col="stop", event_col="event",
          weights_col="w", show_progress=False)
    return float(m.summary.loc["surge", "coef"])

_wmap = dict(zip(cmeas["stay_id"].astype(int), w_vec))
_coef_w = _weighted_surge_hr(cmeas, XC_MEAS[0][_cont_m5], MAIN_ONSET,
                             _cont_m5, _bin_m5, _wmap)

rng = np.random.default_rng(20260629)
_bw = []
_n = len(cmeas)
_cm = cmeas.reset_index(drop=True)
_Xc0 = XC_MEAS[0].reset_index(drop=True)
for b in range(400):
    idx = rng.integers(0, _n, size=_n)
    d_b = _cm.iloc[idx].copy().reset_index(drop=True)
    new = np.arange(len(d_b)); old = d_b["stay_id"].values.copy()
    d_b["stay_id"] = new
    onset_b = {int(new[k]): MAIN_ONSET.get(int(old[k]), np.nan) for k in range(len(d_b))}
    wmap_b = {int(new[k]): _wmap.get(int(old[k]), 1.0) for k in range(len(d_b))}
    Xc_b = _Xc0.iloc[idx][_cont_m5].reset_index(drop=True)
    cf = _weighted_surge_hr(d_b, Xc_b, onset_b, _cont_m5, _bin_m5, wmap_b)
    if cf is not None: _bw.append(cf)
_bw = np.array(_bw)

print("\n[C. weighted surge M5]")
if _coef_w is not None and len(_bw) > 20:
    print(f"  weighted HR = {np.exp(_coef_w):.3f}  "
          f"boot95%CI [{np.exp(np.percentile(_bw,2.5)):.3f}, "
          f"{np.exp(np.percentile(_bw,97.5)):.3f}]  (n_boot={len(_bw)})")
    print(f"  reference, unweighted primary result r_main HR={r_main[0]:.3f} "
          f"[{r_main[1]:.3f},{r_main[2]:.3f}]")
else:
    print("  fit failed or too few bootstrap replicates -> suggests IPW is unstable here (a defensible limitation).")
print("=" * 72)
print("decision guide (data-driven):")
print("  - large ESS reduction (e.g. >30%) plus a weighted CI that comfortably includes 1 ->")
print("    state in the rebuttal that IPW is unstable with 46 events, consistent with the exploratory framing.")
print("  - if the weighted HR and CI resemble the unweighted result, it can be added as a supplementary sensitivity analysis.")
print("=" * 72)

ipw_tab = {"ESS": ESS, "N": len(w), "w_min": float(w.min()), "w_max": float(w.max()),
           "w_cv": float(w.std()/w.mean()),
           "hr_weighted": float(np.exp(_coef_w)) if _coef_w is not None else None,
           "ci_boot": (float(np.exp(np.percentile(_bw,2.5))),
                       float(np.exp(np.percentile(_bw,97.5)))) if len(_bw) > 20 else None}

In [ ]:
import numpy as np, pandas as pd

print("="*72)
print("  [check A] M5 full covariates: m=5 vs m=20 (identifying the Table 2 source)")
print("="*72)

CONT_M5 = CONT_M
BIN_M5  = BIN_M

r5, tab5, _ = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, CONT_M5, BIN_M5, full_return=True)

try:
    _x20 = XC_MEAS_20
    print("  (reusing XC_MEAS_20 from the m=20 cell)")
except NameError:
    _x20 = mice_impute(cmeas, CONT_M5, "event28", "end28_h", m=20)
    print("  (generating a new XC_MEAS_20, m=20)")

r20, tab20, _ = fit_surge_cox(cmeas, _x20, MAIN_ONSET, CONT_M5, BIN_M5, full_return=True)

print(f"\n  surge  m=5 : HR={r5[0]:.3f} [{r5[1]:.3f},{r5[2]:.3f}] p={r5[3]:.3f}")
print(f"  surge  m=20: HR={r20[0]:.3f} [{r20[1]:.3f},{r20[2]:.3f}] p={r20[3]:.3f}")

print(f"\n  {'variable':<18}{'m=5 HR[CI] p':<34}{'m=20 HR[CI] p'}")
def _row(tab, var):
    r = tab[tab['var']==var]
    if len(r)==0: return "  (not found)"
    r=r.iloc[0]
    return f"{r.HR:.3f} [{r.lo:.3f},{r.hi:.3f}] p={r.p:.3f}"
for v in ['surge']+CONT_M5+BIN_M5:
    print(f"  {v:<18}{_row(tab5,v):<34}{_row(tab20,v)}")

print("\n  Verdict: does the manuscript Table 2 (vaso 1.81/SOFA 1.05) match m=5 or m=20 above,")
print("         or neither - check below.")

print("\n"+"="*72)
print("  [check B] landmark 7-statistic crude/full minimum p (verifying the manuscript's 'all P>.13')")
print("="*72)
try:
    _lm = landmark_tab
    pmin_crude = _lm[_lm['model']=='crude']['p'].min()
    pmin_full  = _lm[_lm['model']=='full']['p'].min()
    print(f"  crude min p = {pmin_crude:.3f}")
    print(f"  full  min p = {pmin_full:.3f}")
    print(f"  -> manuscript 'all P>.13': holds for crude={'yes' if pmin_crude>0.13 else 'no'}, "
          f"full={'yes (actual >%.2f)'%pmin_full if pmin_full>0.13 else 'no'}")
except NameError:
    print("  landmark_tab variable not found; check the crude/full minimum p directly in the landmark-cell output.")
    print("  (as reviewed earlier: crude min p=0.143, range from 0.091 / full min from 0.340)")
print("="*72)